In [1]:
from get_write_data_from_dataworks import run_sql
import pandas as pd
import numpy as np

目前的写法排除了每个自然日（dt）没有拨打的催员，后续可以用workhour替代

排除了周日

限制due_date在202411-202504，可以统一调整

In [2]:
start_duemon = 202501
end_duemon = 202507

In [3]:
def generate_bucket(df, bucket_list=[]):
    result = pd.DataFrame()
    for i in bucket_list:
        tmp = df[df['owner_bucket']==i].reset_index(drop=True)
        result = pd.concat([result, tmp], axis=1)
    return result

# cashloan

* 提醒阶段转化率用owing_principal_h3计算

In [4]:
sql = f'''
SELECT  case when user_type = '新客' then '新客'
            when user_type in ('新转化老客','存量老客') then '老客'
            else 'total' end as user_type
        ,TO_CHAR(due_date,'yyyymm') AS due_mon
        ,flag_principal
        ,mob
        ,period_no
        ,case when model_bin >='E' then 'E+' else model_bin end as modelbin_a
        ,SUM(owing_user_cnt) as owing_user_cnt
        ,SUM(owing_principal) as owing_principal
        ,'' as user_cnt_pct
        ,sum(owing_principal)/sum(owing_user_cnt) as owing_principal_avg
        ,AVG(period_seq) as period_seq_avg
        ,SUM(case when datediff(now(),to_date(due_date))>=1 then overdue_principal end)/ SUM(case when datediff(now(),to_date(due_date))>=1 then owing_principal end) AS overdue_rate
        ,1 - SUM(case when datediff(now(),to_date(due_date))>=2 then d2_principal end)/ SUM(case when datediff(now(),to_date(due_date))>=2 then overdue_principal end) AS dpd1_repay
        ,1 - SUM(case when datediff(now(),to_date(due_date))>=6 then d6_principal end)/ SUM(case when datediff(now(),to_date(due_date))>=6 then overdue_principal end) AS dpd5_repay
        ,1 - SUM(case when datediff(now(),to_date(due_date))>=8 then d8_principal end)/ SUM(case when datediff(now(),to_date(due_date))>=8 then overdue_principal end) AS dpd7_repay
        ,1 - SUM(case when datediff(now(),to_date(due_date))>=16 then d16_principal end)/ SUM(case when datediff(now(),to_date(due_date))>=16 then overdue_principal end) AS dpd15_repay
        ,1 - SUM(case when datediff(now(),to_date(due_date))>=31 then d31_principal end)/ SUM(case when datediff(now(),to_date(due_date))>=31 then overdue_principal end) AS dpd30_repay
        ,SUM(case when datediff(now(),to_date(due_date))>=6 then d6_principal end)/ SUM(case when datediff(now(),to_date(due_date))>=6 then owing_principal end) AS dpd5
        ,SUM(case when datediff(now(),to_date(due_date))>=31 then d31_principal end)/ SUM(case when datediff(now(),to_date(due_date))>=31 then owing_principal end) AS dpd30
        
        ,sum(case when is_touch0=2 then owing_user_cnt end)/sum(owing_user_cnt) as connect_rate0
        ,sum(case when action_code0 like '%PTP' then owing_user_cnt end)/sum(owing_user_cnt) as ptp_rate0
        ,case when sum(case when is_touch0=1 and h3_payoff_flag=0 then owing_principal_h3 end)=0 then '/' else sum(case when is_touch0=1  and h3_payoff_flag=0 then owing_principal_h3-d16_principal end)/sum(case when is_touch0=1  and h3_payoff_flag=0 then owing_principal_h3 end) end as disconnect_conversion
        ,case when sum(case when is_touch0=2 and h3_payoff_flag=0 then owing_principal_h3 end) =0 then '/' else sum(case when is_touch0=2 and h3_payoff_flag=0 then owing_principal_h3-d16_principal end)/sum(case when is_touch0=2 and h3_payoff_flag=0 then owing_principal_h3 end) end as connect_conversion
        ,case when sum(case when action_code0 like '%PTP' and h3_payoff_flag=0 then owing_principal_h3 end) = 0 then '/' else sum(case when action_code0 like '%PTP' and h3_payoff_flag=0 then owing_principal_h3-d16_principal end)/sum(case when action_code0 like '%PTP' and h3_payoff_flag=0 then owing_principal_h3 end) end as ptp_conversion
       

        ,case when sum(case when is_touch is not null then owing_user_cnt end)=0 then '/' else sum(case when is_touch=2 then owing_user_cnt end)/sum(case when is_touch is not null then owing_user_cnt end) end as connect_rate
        ,case when sum(case when is_touch is not null then owing_user_cnt end)=0 then '/' else sum(case when action_code like '%PTP' then owing_user_cnt end)/sum(case when is_touch is not null then owing_user_cnt end) end as ptp_rate
        ,case when sum(case when is_touch=1 and d2_payoff_flag=0 then overdue_principal end)=0 then '/' else sum(case when is_touch=1  and d2_payoff_flag=0 then overdue_principal-d16_principal end)/sum(case when is_touch=1  and d2_payoff_flag=0 then overdue_principal end) end as disconnect_conversion
        ,case when sum(case when is_touch=2 and d2_payoff_flag=0 then overdue_principal end) =0 then '/' else sum(case when is_touch=2 and d2_payoff_flag=0 then overdue_principal-d16_principal end)/sum(case when is_touch=2 and d2_payoff_flag=0 then overdue_principal end) end as connect_conversion
        ,case when sum(case when action_code like '%PTP' and d2_payoff_flag=0 then overdue_principal end) = 0 then '/' else sum(case when action_code like '%PTP' and d2_payoff_flag=0 then overdue_principal-d16_principal end)/sum(case when action_code like '%PTP' and d2_payoff_flag=0 then overdue_principal end) end as ptp_conversion
        
FROM    phl_anls.tmp_liujun_phl_ana_09_eoc_sum_daily_temp
WHERE to_char(due_date,'yyyymm')  BETWEEN '{start_duemon}' AND '{end_duemon}'
and model_bin is not null
GROUP BY 
GROUPING SETS (()
             ,(case when user_type = '新客' then '新客'
            when user_type in ('新转化老客','存量老客') then '老客'
            else 'total' end)
             ,(case when user_type = '新客' then '新客'
            when user_type in ('新转化老客','存量老客') then '老客'
            else 'total' end, case when model_bin >='E' then 'E+' else model_bin end
                )
             ,(case when user_type = '新客' then '新客'
            when user_type in ('新转化老客','存量老客') then '老客'
            else 'total' end,TO_CHAR(due_date,'yyyymm'))
             ,(case when user_type = '新客' then '新客'
            when user_type in ('新转化老客','存量老客') then '老客'
            else 'total' end,flag_principal)
            ,(mob)
            ,(case when user_type = '新客' then '新客'
            when user_type in ('新转化老客','存量老客') then '老客'
            else 'total' end,flag_principal,to_char(due_date,'yyyymm'))
            ,(case when user_type = '新客' then '新客'
            when user_type in ('新转化老客','存量老客') then '老客'
            else 'total' end, case when model_bin >='E' then 'E+' else model_bin end, to_char(due_date,'yyyymm'))
            ,(case when user_type = '新客' then '新客'
            when user_type in ('新转化老客','存量老客') then '老客'
            else 'total' end, period_no, to_char(due_date,'yyyymm'))
            ,(case when user_type = '新客' then '新客'
            when user_type in ('新转化老客','存量老客') then '老客'
            else 'total' end, period_no)
            
            )
'''
df = run_sql(sql)

log_view_address: http://logview.odps.aliyun.com/logview/?h=https://service.ap-southeast-1-vpc.maxcompute.aliyun-inc.com/api&p=phl_anls&i=20260203114352552g5y330u1al7&token=SWtMNHVOaFRHS1JiZnIvQ1dxVS9TbW5tV21BPSxPRFBTX09CTzpwNF8yNjE4MzAxNzM0MTg2MTQ1NDcsMTc3MjcxMTAzMix7IlN0YXRlbWVudCI6W3siQWN0aW9uIjpbIm9kcHM6UmVhZCJdLCJFZmZlY3QiOiJBbGxvdyIsIlJlc291cmNlIjpbImFjczpvZHBzOio6cHJvamVjdHMvcGhsX2FubHMvaW5zdGFuY2VzLzIwMjYwMjAzMTE0MzUyNTUyZzV5MzMwdTFhbDciXX1dLCJWZXJzaW9uIjoiMSJ9


In [5]:
df_total_cl = df[(df['flag_principal'].isna()) & (df['due_mon'].isna()) & (df['mob'].isna()) & (df['modelbin_a'].isna())&(df['period_no'].isna())].drop(columns=['due_mon','flag_principal','mob']).sort_values(by='user_type')
df_duemon = df[df['due_mon'].notna()&df['flag_principal'].isna()&df['modelbin_a'].isna()&df['period_no'].isna()].drop(columns=['flag_principal','mob']).sort_values(by=['user_type', 'due_mon'],ascending=[True,False])
df_principal = df[df['flag_principal'].notna()&df['due_mon'].isna()].drop(columns=['due_mon','mob','modelbin_a','period_no']).sort_values(by=['user_type', 'flag_principal'])
df_mob = df[df['mob'].notna()].drop(columns=['flag_principal','modelbin_a','due_mon','period_no','user_type']).sort_values(by='mob')
df_modelbina = df[(df['modelbin_a'].notna()) & (df['due_mon'].isna())].drop(columns=['flag_principal','mob','period_no','due_mon']).sort_values(by=['user_type','modelbin_a'])
df_period =  df[(df['period_no'].notna()) & (df['due_mon'].isna())].drop(columns=['flag_principal','modelbin_a','mob','due_mon']).sort_values(by=['user_type','period_no'])

df_principal_duemon = df[(df['flag_principal'].notna()) & (df['due_mon'].notna())].drop(columns=['mob']).groupby(by='user_type')
df_principal_duemon_user = df_principal_duemon.apply(lambda x: pd.pivot_table(x,index='flag_principal',columns='due_mon',values='owing_user_cnt',aggfunc='sum'))
df_principal_duemon_user = df_principal_duemon_user.apply(lambda x:x.div(x.sum()))
df_principal_duemon_overdue = df_principal_duemon.apply(lambda x: pd.pivot_table(x,index='flag_principal',columns='due_mon',values='overdue_rate',aggfunc='sum'))
df_principal_duemon_dpd5 = df_principal_duemon.apply(lambda x: pd.pivot_table(x,index='flag_principal',columns='due_mon',values='dpd5',aggfunc='sum'))
df_principal_duemon_result = pd.concat([df_principal_duemon_user,df_principal_duemon_overdue,df_principal_duemon_dpd5],axis=1)

df_modelbina_duemon = df[(df['modelbin_a'].notna()) & (df['due_mon'].notna())].drop(columns=['mob','flag_principal']).groupby(by='user_type')
df_modelbina_duemon_user = df_modelbina_duemon.apply(lambda x: pd.pivot_table(x,index='modelbin_a',columns='due_mon',values='owing_user_cnt',aggfunc='sum'))
df_modelbina_duemon_user = df_modelbina_duemon_user.apply(lambda x:x.div(x.sum()))
df_modelbina_duemon_overdue = df_modelbina_duemon.apply(lambda x: pd.pivot_table(x,index='modelbin_a',columns='due_mon',values='overdue_rate',aggfunc='sum'))
df_modelbina_duemon_dpd5 = df_modelbina_duemon.apply(lambda x: pd.pivot_table(x,index='modelbin_a',columns='due_mon',values='dpd5',aggfunc='sum'))
df_modelbina_duemon_result = pd.concat([df_modelbina_duemon_user,df_modelbina_duemon_overdue,df_modelbina_duemon_dpd5],axis=1)

df_period_duemon = df[(df['period_no'].notna()) & (df['due_mon'].notna())].drop(columns=['mob','modelbin_a','flag_principal']).groupby(by='user_type')
df_period_duemon_user = df_period_duemon.apply(lambda x: pd.pivot_table(x,index='period_no',columns='due_mon',values='owing_user_cnt',aggfunc='sum'))
df_period_duemon_user = df_period_duemon_user.apply(lambda x:x.div(x.sum()))
df_period_duemon_overdue = df_period_duemon.apply(lambda x: pd.pivot_table(x,index='period_no',columns='due_mon',values='overdue_rate',aggfunc='sum'))
df_period_duemon_dpd5 = df_period_duemon.apply(lambda x: pd.pivot_table(x,index='period_no',columns='due_mon',values='dpd5',aggfunc='sum'))
df_period_duemon_result = pd.concat([df_period_duemon_user,df_period_duemon_overdue,df_period_duemon_dpd5],axis=1)

In [6]:
df_principal_duemon_result

due_mon                     202501    202502    202503    202504    202505  \
user_type flag_principal                                                     
新客        1) 0-1000       0.008626  0.007470  0.006051  0.005152  0.004818   
          2) 1001-2000    0.008898  0.010081  0.009779  0.010161  0.010664   
          3) 2001-3000    0.011971  0.013098  0.011930  0.010773  0.010664   
          4) 3001-4000    0.002192  0.002730  0.002617  0.002726  0.003216   
          5) 4001-5000    0.004347  0.004796  0.003888  0.003451  0.003043   
          6) >5000        0.006946  0.010215  0.010060  0.011633  0.015341   
老客        1) 0-1000       0.295053  0.307705  0.318061  0.320106  0.320754   
          2) 1001-2000    0.328947  0.325437  0.328504  0.328716  0.328246   
          3) 2001-3000    0.149015  0.142892  0.140482  0.139471  0.137737   
          4) 3001-4000    0.065309  0.061981  0.059909  0.058269  0.057575   
          5) 4001-5000    0.039333  0.037365  0.035717  0.035688  0.035465   
          6) >5000        0.079362  0.076230  0.073002  0.073854  0.072476   

due_mon                     202506    202507                202501  \
user_type flag_principal                                             
新客        1) 0-1000       0.004906  0.005774  0.232908206281899952   
          2) 1001-2000    0.009639  0.009799  0.141447828340722004   
          3) 2001-3000    0.009518  0.009908  0.184492053738418002   
          4) 3001-4000    0.003198  0.003178  0.170566154667781596   
          5) 4001-5000    0.002717  0.002794  0.196593803286783151   
          6) >5000        0.013960  0.014702  0.138528745576563749   
老客        1) 0-1000       0.316842  0.314298   0.08116740001705255   
          2) 1001-2000    0.331324  0.333146  0.075994538201355812   
          3) 2001-3000    0.140154  0.140374  0.072892871225929551   
          4) 3001-4000    0.057992  0.057792  0.075936952427441624   
          5) 4001-5000    0.035360  0.034954  0.072803512133408267   
          6) >5000        0.074392  0.073281  0.094775611632706129   

due_mon                                 202502                202503  ...  \
user_type flag_principal                                              ...   
新客        1) 0-1000       0.211786830270120115   0.20367146964804909  ...   
          2) 1001-2000     0.12650664344334174  0.147026448801192201  ...   
          3) 2001-3000    0.169473712558103876  0.178115823415567613  ...   
          4) 3001-4000    0.171984238705408169  0.155269271034042554  ...   
          5) 4001-5000    0.170565456449923967  0.184166096310017027  ...   
          6) >5000        0.129473211718127412  0.138495872851122323  ...   
老客        1) 0-1000       0.080651643644543784  0.077311144614943528  ...   
          2) 1001-2000    0.077883152170469593  0.074276947637953169  ...   
          3) 2001-3000    0.076429729020348439  0.072873580101496895  ...   
          4) 3001-4000     0.07453724820369517  0.073913822340163138  ...   
          5) 4001-5000    0.073704877416805357  0.073157087276407505  ...   
          6) >5000         0.08887797398472336  0.085112974644317457  ...   

due_mon                                 202505                202506  \
user_type flag_principal                                               
新客        1) 0-1000       0.218856369816496705  0.220542602861607327   
          2) 1001-2000     0.14869263689646348  0.147954544919059212   
          3) 2001-3000    0.184328012076475295  0.179629021636752368   
          4) 3001-4000    0.169804169643158736   0.14560990534017443   
          5) 4001-5000    0.150492777097145156  0.151050131463979733   
          6) >5000        0.153585569034211113  0.161369576641568565   
老客        1) 0-1000       0.079368563245256922  0.080109302184913468   
          2) 1001-2000    0.078615469556082847   0.08019685375264917   
          3) 2001-3000    0.078953659586230717  0.080786800395738578   
          4) 3001-4000    0.079064423328761599   0.0801

In [7]:
# CL MODELBINC
sql = f'''
SELECT  case when user_type = '新客' then '新客'
            when user_type in ('新转化老客','存量老客') then '老客'
            else 'total' end as user_type
        ,flag_collect_bin as modelbin_c
        ,to_char(due_date,'yyyymm') AS due_mon
        ,count(distinct user_id) as owing_user_cnt
        ,COUNT(DISTINCT CASE WHEN overdue_principal > 0 then user_id end) as overdue_user_cnt
        ,SUM(principal) as owing_principal
        ,'' as user_percentage
        ,sum(principal)/count(distinct user_id) as owing_principal_avg
        ,AVG(period_seq) as period_seq_avg
        ,SUM(case when datediff(now(),to_date(due_date))>=1 then overdue_principal end)/ SUM(case when datediff(now(),to_date(due_date))>=1 then owing_principal end) AS overdue_rate
        ,1 - SUM(case when datediff(now(),to_date(due_date))>=2 then d2_principal end)/ SUM(case when datediff(now(),to_date(due_date))>=2 then overdue_principal end) AS dpd1_repay
        ,1 - SUM(case when datediff(now(),to_date(due_date))>=6 then d6_principal end)/ SUM(case when datediff(now(),to_date(due_date))>=6 then overdue_principal end) AS dpd5_repay
        ,1 - SUM(case when datediff(now(),to_date(due_date))>=8 then d8_principal end)/ SUM(case when datediff(now(),to_date(due_date))>=8 then overdue_principal end) AS dpd7_repay
        ,1 - SUM(case when datediff(now(),to_date(due_date))>=16 then d16_principal end)/ SUM(case when datediff(now(),to_date(due_date))>=16 then overdue_principal end) AS dpd15_repay
        ,1 - SUM(case when datediff(now(),to_date(due_date))>=31 then d31_principal end)/ SUM(case when datediff(now(),to_date(due_date))>=31 then overdue_principal end) AS dpd30_repay
        ,SUM(case when datediff(now(),to_date(due_date))>=6 then d6_principal end)/ SUM(case when datediff(now(),to_date(due_date))>=6 then owing_principal end) AS dpd5
        ,SUM(case when datediff(now(),to_date(due_date))>=31 then d31_principal end)/ SUM(case when datediff(now(),to_date(due_date))>=31 then owing_principal end) AS dpd30
        
        ,count(distinct case when is_touch0=2 then user_id end)/count(distinct user_id) as connect_rate0
        ,count(distinct case when action_code0 like '%PTP' then user_id end)/count(distinct user_id) as ptp_rate0
        ,case when sum(case when is_touch0=1 and h3_payoff_flag=0 then owing_principal_h3 end)=0 then '/' else sum(case when is_touch0=1  and h3_payoff_flag=0 then owing_principal_h3-d16_principal end)/sum(case when is_touch0=1  and h3_payoff_flag=0 then owing_principal_h3 end) end as disconnect_conversion
        ,case when sum(case when is_touch0=2 and h3_payoff_flag=0 then owing_principal_h3 end) =0 then '/' else sum(case when is_touch0=2 and h3_payoff_flag=0 then owing_principal_h3-d16_principal end)/sum(case when is_touch0=2 and h3_payoff_flag=0 then owing_principal_h3 end) end as connect_conversion
        ,case when sum(case when action_code0 like '%PTP' and h3_payoff_flag=0 then owing_principal_h3 end) = 0 then '/' else sum(case when action_code0 like '%PTP' and h3_payoff_flag=0 then owing_principal_h3-d16_principal end)/sum(case when action_code0 like '%PTP' and h3_payoff_flag=0 then owing_principal_h3 end) end as ptp_conversion
       
        ,count(distinct case when is_touch=2 then user_id end)/count(distinct case when is_touch is not null then user_id end) as connect_rate
        ,count(distinct case when action_code like '%PTP' then user_id end)/count(distinct case when is_touch is not null then user_id end) as ptp_rate
        ,case when sum(case when is_touch=1 and d2_payoff_flag=0 then overdue_principal end)=0 then '/' else sum(case when is_touch=1  and d2_payoff_flag=0 then overdue_principal-d16_principal end)/sum(case when is_touch=1  and d2_payoff_flag=0 then overdue_principal end) end as disconnect_conversion
        ,case when sum(case when is_touch=2 and d2_payoff_flag=0 then overdue_principal end) =0 then '/' else sum(case when is_touch=2 and d2_payoff_flag=0 then overdue_principal-d16_principal end)/sum(case when is_touch=2 and d2_payoff_flag=0 then overdue_principal end) end as connect_conversion
        ,case when sum(case when action_code like '%PTP' and d2_payoff_flag=0 then overdue_principal end) = 0 then '/' else sum(case when action_code like '%PTP' and d2_payoff_flag=0 then overdue_principal-d16_principal end)/sum(case when action_code like '%PTP' and d2_payoff_flag=0 then overdue_principal end) end as ptp_conversion
FROM    phl_anls.tmp_zyt_cl_vintage
WHERE to_char(due_date,'yyyymm')  BETWEEN '202501' AND '202507'
and flag_collect_bin <> 'err'
GROUP BY 
GROUPING SETS (
            (case when user_type = '新客' then '新客'
            when user_type in ('新转化老客','存量老客') then '老客'
            else 'total' end
            ,flag_collect_bin)
            ,(case when user_type = '新客' then '新客'
            when user_type in ('新转化老客','存量老客') then '老客'
            else 'total' end
            ,flag_collect_bin,to_char(due_date,'yyyymm'))

)

'''
df = run_sql(sql)

log_view_address: http://logview.odps.aliyun.com/logview/?h=https://service.ap-southeast-1-vpc.maxcompute.aliyun-inc.com/api&p=phl_anls&i=20260203114412749go0o4womh2&token=b1h1c0N3RFA2azRmZTVXTmpLMDZ1MWJWRUxVPSxPRFBTX09CTzpwNF8yNjE4MzAxNzM0MTg2MTQ1NDcsMTc3MjcxMTA1Mix7IlN0YXRlbWVudCI6W3siQWN0aW9uIjpbIm9kcHM6UmVhZCJdLCJFZmZlY3QiOiJBbGxvdyIsIlJlc291cmNlIjpbImFjczpvZHBzOio6cHJvamVjdHMvcGhsX2FubHMvaW5zdGFuY2VzLzIwMjYwMjAzMTE0NDEyNzQ5Z28wbzR3b21oMiJdfV0sIlZlcnNpb24iOiIxIn0=


In [8]:
df_modelbinc = df[(df['modelbin_c'].notna()) & (df['modelbin_c']!='err') & (df['due_mon'].isna())].sort_values(by=['user_type','modelbin_c'])

df_modelbinc_duemon = df[df['modelbin_c'].notna() & df['due_mon'].notna()].groupby(by='user_type')
df_modelbinc_duemon_user = df_modelbinc_duemon.apply(lambda x: pd.pivot_table(x,index='modelbin_c',columns='due_mon',values='overdue_user_cnt',aggfunc='sum'))
df_modelbinc_duemon_user = df_modelbinc_duemon_user.apply(lambda x:x.div(x.sum()))
df_modelbinc_duemon_overdue = df_modelbinc_duemon.apply(lambda x: pd.pivot_table(x,index='modelbin_c',columns='due_mon',values='overdue_rate',aggfunc='sum'))
df_modelbinc_duemon_dpd5 = df_modelbinc_duemon.apply(lambda x: pd.pivot_table(x,index='modelbin_c',columns='due_mon',values='dpd5',aggfunc='sum'))
df_modelbinc_duemon_result = pd.concat([df_modelbinc_duemon_user,df_modelbinc_duemon_overdue,df_modelbinc_duemon_dpd5],axis=1)

In [9]:
df[(df['modelbin_c'].notna())& (df['due_mon'].isna())].sort_values(by=['user_type','modelbin_c'])

,user_type,modelbin_c,due_mon,owing_user_cnt,overdue_user_cnt,owing_principal,user_percentage,owing_principal_avg,period_seq_avg,overdue_rate,...,connect_rate0,ptp_rate0,disconnect_conversion,connect_conversion,ptp_conversion,connect_rate,ptp_rate,disconnect_conversion2,connect_conversion2,ptp_conversion2
45,新客,A,None,21217,3156,97571911,,4598.760946410896922279,1.428476,2.608859668384557014,...,0.581138,0.451525,0.886640695709630544,0.959862274384706394,0.969712962244562526,0.478657,0.095417,0.386484935896222586,0.448777409725008868,0.642077895175423626
14,新客,B,None,33780,7586,135655486,,4015.852161042036708111,1.275105,2.277469299266110765,...,0.547395,0.404677,0.815861592427828303,0.929160620937702182,0.948190197152865207,0.462204,0.076903,0.344266476962742137,0.414872513392857498,0.630957484049187692
18,新客,C,None,21634,6262,71454809,,3302.894009429601553111,1.063281,1.957253896760461124,...,0.526810,0.379264,0.748328623779234003,0.890553838905535967,0.921166589508597469,0.475608,0.075163,0.271344100883322333,0.344958035063678958,0.577524306533353159
24,新客,D,None,37087,12330,135662614,,3657.955995362256316229,1.180037,1.943171151645649531,...,0.511635,0.368350,0.713519668130097305,0.886197006408359875,0.911693267771204193,0.458539,0.065967,0.282059235772130819,0.352253169141493047,0.555239135174127737
28,新客,E,None,37083,20007,124293051,,3351.75285171102661597,1.139760,1.502546262459227959,...,0.463015,0.319715,0.480799508442183041,0.764758447472908222,0.822094860233783645,0.472394,0.055860,0.176660209673956012,0.223083812502965256,0.435362710337361284
61,老客,A,None,194666,45322,3524880320.67,,18107.323932633330936065,3.109719,4.051922251949496539,...,0.630994,0.518432,0.977928100677430938,0.992264343612486058,0.993805800181603285,0.388351,0.119418,0.462386030129688519,0.553761952684992466,0.799164251915558988
35,老客,B,None,204274,46613,1785352043.13,,8739.986699873699051274,3.155731,3.505151474657115429,...,0.583368,0.462026,0.959090022280068666,0.984148246701036642,0.988664053886141131,0.367593,0.097903,0.444409771230492562,0.515542601209267015,0.79600369028668639
68,老客,C,None,192386,54270,1304104466.99,,6778.582989354734752009,3.163831,3.5137803958984191,...,0.539961,0.416402,0.942555990244360522,0.977330686096980938,0.983211161648727663,0.366417,0.089238,0.447258673207112346,0.50641079083644744,0.789070666566116806
40,老客,D,None,182252,73351,1187419093.37,,6515.2596041195707043,3.180292,3.777527341588427673,...,0.506590,0.382383,0.921383188242862552,0.965032317343866684,0.973283381875727587,0.378254,0.087175,0.487660307444366739,0.54245652125739984,0.799257188874106344
8,老客,E,None,166372,105868,1214986971.81,,7302.83324002837015844,3.307595,4.464301843792466931,...,0.474136,0.328667,0.867092841373672488,0.927187884211369511,0.941382965332924659,0.467438,0.098399,0.567553238467699147,0.603263362293338607,0.812438942616822918


### 1.2+3 CL过程数据

In [10]:
# start_duemon = 202411
# end_duemon = 202507
# !!NEW!!
# user_type
sql = f'''
    SELECT  owner_bucket
            ,user_type
            
            ,sum(owner_cnt) / count(distinct dt) AS owner_cnt_avg
            ,SUM(owing_case_cnt) / SUM(owner_cnt) AS owing_case_avg
            ,sum(call_connect_times) / sum(owner_cnt) AS call_connect_times_pp_avg
            ,sum(call_case_cnt) / sum(owing_case_cnt) as call_case_pct
            ,'' as call_duration_avg_owner
            ,case when sum(owing_case_cnt)>0 then sum(comm_times)/(sum(owing_case_cnt)-sum(own_uncomm_case_cnt)) end as cover_times
            ,SUM(call_times) / SUM(owing_case_cnt) AS call_times_avg
            ,SUM(call_duration) / SUM(owing_case_cnt) AS call_duration_avg
            ,SUM(call_contact_mobile_times) / sum(owing_case_cnt) as call_contact_mobile_times_avg
            
            ,SUM(call_connect_times) / SUM(owing_case_cnt) AS case_connect_times_avg
            ,SUM(call_connect_case_cnt) / SUM(call_case_cnt) AS case_connect_rate
            ,SUM(call_connect_times) / SUM(call_times) AS call_connect_rate
            ,sum(call_connect_contact_mobile_times) / sum(call_contact_mobile_times) as contact_call_connect_rate
            ,SUM(connect_duration) / SUM(call_connect_case_cnt) as connect_duration_case_avg
            
    FROM    (
                SELECT  owner_bucket
                        ,user_type
                        ,dt
                        ,datediff(CAST(max(last_comm_time) AS DATETIME), CAST(min(first_comm_time) AS DATETIME), 'hour') AS comm_hour
                        ,count(distinct case when work_hour >= 8 then owner_name end) AS owner_cnt
                        ,COUNT(DISTINCT global_case_id) AS case_cnt
                        ,SUM(CASE    WHEN (case_status <> 2 OR SUBSTR(trans_time,1,10) <> dt) THEN 1 ELSE 0 END) AS owing_case_cnt
                        ,SUM(call_times) AS call_times
                        ,sum(comm_times) as comm_times
                        ,SUM(call_duration) / 60 AS call_duration
                        ,SUM(call_connect_times) AS call_connect_times
                        ,SUM(call_billsec) / 60 AS connect_duration
                        ,SUM(call_contact_mobile_times) as call_contact_mobile_times
                        ,SUM(call_connect_contact_mobile_times) as call_connect_contact_mobile_times
                        ,SUM(CASE    WHEN is_case_comm = 0 THEN 1 ELSE 0 END) as own_uncomm_case_cnt
                        ,COUNT(DISTINCT 
                              CASE    WHEN ((call_connect_times) > 0) AND (case_status <> 2 OR SUBSTR(trans_time,1,10) <> dt)
                              THEN global_case_id END
                        ) AS call_connect_case_cnt
                        ,COUNT(DISTINCT CASE    WHEN ((call_times) > 0) AND (case_status <> 2 OR SUBSTR(trans_time,1,10) <> dt) 
                        THEN global_case_id END) AS call_case_cnt
                FROM    (select *
                        ,sum(call_times) over (partition by dt) as call_times_check
                        -- ,sum(call_times) over (partition by dt, owner_id) as call_times_agent_check
                        ,MAX(HOUR(last_call_time) + MINUTE(last_call_time) / 60) OVER (PARTITION BY dt,owner_id)-MIN(HOUR(first_call_time) + MINUTE(first_call_time) / 60) OVER (PARTITION BY dt,owner_id) as work_hour
                        from phl_data.dws_fact_coll_case_process_info_daily
                        WHERE   dt >= '2024-10-20'
                        AND     TO_CHAR(DATEADD(TO_DATE(dt),-overdue_days_dt,'day'),'yyyymm')  BETWEEN '{start_duemon}' AND '{end_duemon}'
                        AND     owner_bucket IN ('S0','S1','S2','M1')
                        )
                where call_times_check > 2000
                GROUP BY owner_bucket
                         ,user_type
                         ,dt
                         ,owner_id
            ) a
    GROUP BY owner_bucket
             ,user_type
    ;
'''
df = run_sql(sql)
df_total_cl_process = generate_bucket(df.sort_values('user_type'),['S0','S1','S2','M1'])

df_total_cl = pd.concat([df_total_cl.reset_index(drop=True),df_total_cl_process],axis=1)
df_total_cl['tag'] = 'Cashloan'

log_view_address: http://logview.odps.aliyun.com/logview/?h=https://service.ap-southeast-1-vpc.maxcompute.aliyun-inc.com/api&p=phl_anls&i=20260203114427282g83oaxwawo&token=WUk5L3NxOUJUdjFSODVENzByTWRkZ2J6WERzPSxPRFBTX09CTzpwNF8yNjE4MzAxNzM0MTg2MTQ1NDcsMTc3MjcxMTA2Nyx7IlN0YXRlbWVudCI6W3siQWN0aW9uIjpbIm9kcHM6UmVhZCJdLCJFZmZlY3QiOiJBbGxvdyIsIlJlc291cmNlIjpbImFjczpvZHBzOio6cHJvamVjdHMvcGhsX2FubHMvaW5zdGFuY2VzLzIwMjYwMjAzMTE0NDI3MjgyZzgzb2F4d2F3byJdfV0sIlZlcnNpb24iOiIxIn0=


In [11]:
df_total_cl

,user_type,period_no,modelbin_a,owing_user_cnt,owing_principal,user_cnt_pct,owing_principal_avg,period_seq_avg,overdue_rate,dpd1_repay,...,cover_times,call_times_avg,call_duration_avg,call_contact_mobile_times_avg,case_connect_times_avg,case_connect_rate,call_connect_rate,contact_call_connect_rate,connect_duration_case_avg,tag
0,新客,NaN,None,292073,1045206460,,3578.579533198892057807,1.205373,0.161832728789295849,0.113325690382127795,...,5.609641,4.298958,1.662558,1.564771,0.160125,0.162071,0.037247,0.063372,1.137229,Cashloan
1,老客,NaN,None,6144170,13878344458,,2258.782627759323065605,3.163957,0.083246659419382456,0.355383439556598208,...,7.709454,6.281897,2.308964,1.983987,0.144290,0.138552,0.022969,0.049119,1.147013,Cashloan
2,None,NaN,None,6436243,14923550918,,2318.674251112022961221,3.075229,0.088750622103784214,0.324470247265305531,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Cashloan


In [12]:
# !!NEW!!
# user_type+due_mon
sql = f'''
    SELECT  owner_bucket
            ,user_type
            ,due_mon
            ,sum(owner_cnt) / count(distinct dt) AS owner_cnt_avg
            ,SUM(owing_case_cnt) / SUM(owner_cnt) AS owing_case_avg
            ,sum(call_connect_times) / sum(owner_cnt) AS call_connect_times_pp_avg
            ,sum(call_case_cnt) / sum(owing_case_cnt) as call_case_pct
            ,'' as call_duration_avg_owner
            ,case when sum(owing_case_cnt)>0 then sum(comm_times)/(sum(owing_case_cnt)-sum(own_uncomm_case_cnt)) end as cover_times
            ,SUM(call_times) / SUM(owing_case_cnt) AS call_times_avg
            ,SUM(call_duration) / SUM(owing_case_cnt) AS call_duration_avg
            ,SUM(call_contact_mobile_times) / sum(owing_case_cnt) as call_contact_mobile_times_avg
            
            ,SUM(call_connect_times) / SUM(owing_case_cnt) AS case_connect_times_avg
            ,SUM(call_connect_case_cnt) / SUM(call_case_cnt) AS case_connect_rate
            ,SUM(call_connect_times) / SUM(call_times) AS call_connect_rate
            ,sum(call_connect_contact_mobile_times) / sum(call_contact_mobile_times) as contact_call_connect_rate
            ,SUM(connect_duration) / SUM(call_connect_case_cnt) as connect_duration_case_avg
    FROM    (
                SELECT  owner_bucket
                        ,user_type
                        ,dt
                        ,TO_CHAR(DATEADD(TO_DATE(dt),-overdue_days_dt,'day'),'yyyymm') as due_mon
                        ,datediff(CAST(max(last_comm_time) AS DATETIME), CAST(min(first_comm_time) AS DATETIME), 'hour') AS comm_hour
                        ,count(distinct case when work_hour >= 8 then owner_name end) AS owner_cnt
                        ,COUNT(DISTINCT global_case_id) AS case_cnt
                        ,SUM(CASE    WHEN (case_status <> 2 OR SUBSTR(trans_time,1,10) <> dt) THEN 1 ELSE 0 END) AS owing_case_cnt
                        ,SUM(call_times) AS call_times
                        ,sum(comm_times) as comm_times
                        ,SUM(call_duration) / 60 AS call_duration
                        ,SUM(call_connect_times) AS call_connect_times
                        ,SUM(call_billsec) / 60 AS connect_duration
                        ,SUM(call_contact_mobile_times) as call_contact_mobile_times
                        ,SUM(call_connect_contact_mobile_times) as call_connect_contact_mobile_times
                        ,SUM(CASE    WHEN is_case_comm = 0 THEN 1 ELSE 0 END) as own_uncomm_case_cnt
                        ,COUNT(DISTINCT 
                              CASE    WHEN ((call_connect_times) > 0) AND (case_status <> 2 OR SUBSTR(trans_time,1,10) <> dt)
                              THEN global_case_id END
                        ) AS call_connect_case_cnt
                        ,COUNT(DISTINCT CASE    WHEN ((call_times) > 0) AND (case_status <> 2 OR SUBSTR(trans_time,1,10) <> dt) 
                        THEN global_case_id END) AS call_case_cnt
                FROM    (select *
                        ,sum(call_times) over (partition by dt) as call_times_check
                        -- ,sum(call_times) over (partition by dt, owner_id) as call_times_agent_check
                        ,MAX(HOUR(last_call_time) + MINUTE(last_call_time) / 60) OVER (PARTITION BY dt,owner_id)-MIN(HOUR(first_call_time) + MINUTE(first_call_time) / 60) OVER (PARTITION BY dt,owner_id) as work_hour
                        from phl_data.dws_fact_coll_case_process_info_daily
                        WHERE   dt >= '2024-10-20'
                        AND     TO_CHAR(DATEADD(TO_DATE(dt),-overdue_days_dt,'day'),'yyyymm')  BETWEEN '{start_duemon}' AND '{end_duemon}'
                        AND     owner_bucket IN ('S0','S1','S2','M1')
                        )
                where call_times_check > 2000
                GROUP BY owner_bucket
                         ,user_type
                         ,dt
                         ,TO_CHAR(DATEADD(TO_DATE(dt),-overdue_days_dt,'day'),'yyyymm')
            ) a
    GROUP BY owner_bucket
             ,user_type
             ,due_mon
    ;
'''
df = run_sql(sql)
df_duemon_process = generate_bucket(df.sort_values(by=['user_type', 'due_mon'],ascending=[True,False]), ['S0','S1','S2','M1'])

log_view_address: http://logview.odps.aliyun.com/logview/?h=https://service.ap-southeast-1-vpc.maxcompute.aliyun-inc.com/api&p=phl_anls&i=20260203114444792gcyeuyd02l7&token=RGxvVHlCRTJJTjJzTXJjTW8wbk5zbjdKeU93PSxPRFBTX09CTzpwNF8yNjE4MzAxNzM0MTg2MTQ1NDcsMTc3MjcxMTA4NCx7IlN0YXRlbWVudCI6W3siQWN0aW9uIjpbIm9kcHM6UmVhZCJdLCJFZmZlY3QiOiJBbGxvdyIsIlJlc291cmNlIjpbImFjczpvZHBzOio6cHJvamVjdHMvcGhsX2FubHMvaW5zdGFuY2VzLzIwMjYwMjAzMTE0NDQ0NzkyZ2N5ZXV5ZDAybDciXX1dLCJWZXJzaW9uIjoiMSJ9


In [13]:
# !!NEW!!
# user_type+flag_principal
sql = f'''
    SELECT  owner_bucket
            ,user_type
            ,flag_owing_principal_alloc
            ,sum(owner_cnt) / count(distinct dt) AS owner_cnt_avg
            ,SUM(owing_case_cnt) / SUM(owner_cnt) AS owing_case_avg
            ,sum(call_connect_times) / sum(owner_cnt) AS call_connect_times_pp_avg
            ,sum(call_case_cnt) / sum(owing_case_cnt) as call_case_pct
            ,'' as call_duration_avg_owner
            ,case when sum(owing_case_cnt)>0 then sum(comm_times)/(sum(owing_case_cnt)-sum(own_uncomm_case_cnt)) end as cover_times
            ,SUM(call_times) / SUM(owing_case_cnt) AS call_times_avg
            ,SUM(call_duration) / SUM(owing_case_cnt) AS call_duration_avg
            ,SUM(call_contact_mobile_times) / sum(owing_case_cnt) as call_contact_mobile_times_avg
            
            ,SUM(call_connect_times) / SUM(owing_case_cnt) AS case_connect_times_avg
            ,SUM(call_connect_case_cnt) / SUM(call_case_cnt) AS case_connect_rate
            ,SUM(call_connect_times) / SUM(call_times) AS call_connect_rate
            ,sum(call_connect_contact_mobile_times) / sum(call_contact_mobile_times) as contact_call_connect_rate
            ,SUM(connect_duration) / SUM(call_connect_case_cnt) as connect_duration_case_avg
    FROM    (
                SELECT  owner_bucket
                        ,user_type
                        ,dt
                        ,case when owing_principal_alloc < 1500 then '1. 0-1500'
                                when owing_principal_alloc < 2500 then '2. 1500-2500'
                                when owing_principal_alloc < 3500 then '3, 2500-3500'
                                when owing_principal_alloc < 5000 then '4, 3500-5000'
                                when owing_principal_alloc < 8000 then '5, 5000-8000'
                                else '6. 8000+' end as flag_owing_principal_alloc
                        ,count(distinct case when work_hour >= 8 then owner_name end) AS owner_cnt
                        ,COUNT(DISTINCT global_case_id) AS case_cnt
                        ,SUM(CASE    WHEN (case_status <> 2 OR SUBSTR(trans_time,1,10) <> dt) THEN 1 ELSE 0 END) AS owing_case_cnt
                        ,SUM(call_times) AS call_times
                        ,sum(comm_times) as comm_times
                        ,SUM(call_duration) / 60 AS call_duration
                        ,SUM(call_connect_times) AS call_connect_times
                        ,SUM(call_billsec) / 60 AS connect_duration
                        ,SUM(call_contact_mobile_times) as call_contact_mobile_times
                        ,SUM(call_connect_contact_mobile_times) as call_connect_contact_mobile_times
                        ,SUM(CASE    WHEN is_case_comm = 0 THEN 1 ELSE 0 END) as own_uncomm_case_cnt
                        ,COUNT(DISTINCT 
                              CASE    WHEN ((call_connect_times) > 0) AND (case_status <> 2 OR SUBSTR(trans_time,1,10) <> dt)
                              THEN global_case_id END
                        ) AS call_connect_case_cnt
                        ,COUNT(DISTINCT CASE    WHEN ((call_times) > 0) AND (case_status <> 2 OR SUBSTR(trans_time,1,10) <> dt) 
                        THEN global_case_id END) AS call_case_cnt
                FROM    (select *
                        ,sum(call_times) over (partition by dt) as call_times_check
                        -- ,sum(call_times) over (partition by dt, owner_id) as call_times_agent_check
                        ,MAX(HOUR(last_call_time) + MINUTE(last_call_time) / 60) OVER (PARTITION BY dt,owner_id)-MIN(HOUR(first_call_time) + MINUTE(first_call_time) / 60) OVER (PARTITION BY dt,owner_id) as work_hour
                        from phl_data.dws_fact_coll_case_process_info_daily
                        WHERE   dt >= '2024-10-20'
                        AND     TO_CHAR(DATEADD(TO_DATE(dt),-overdue_days_dt,'day'),'yyyymm')  BETWEEN '{start_duemon}' AND '{end_duemon}'
                        AND     owner_bucket IN ('S0','S1','S2','M1')
                        )
                where call_times_check > 2000
                GROUP BY owner_bucket
                         ,user_type
                         ,dt
                         ,case when owing_principal_alloc < 1500 then '1. 0-1500'
                                when owing_principal_alloc < 2500 then '2. 1500-2500'
                                when owing_principal_alloc < 3500 then '3, 2500-3500'
                                when owing_principal_alloc < 5000 then '4, 3500-5000'
                                when owing_principal_alloc < 8000 then '5, 5000-8000'
                                else '6. 8000+' end
            ) a
    GROUP BY owner_bucket
             ,user_type
             ,flag_owing_principal_alloc
    ;
'''
df = run_sql(sql)
df_principal_process = generate_bucket(df.sort_values(by=['user_type', 'flag_owing_principal_alloc']), ['S0','S1','S2','M1'])

log_view_address: http://logview.odps.aliyun.com/logview/?h=https://service.ap-southeast-1-vpc.maxcompute.aliyun-inc.com/api&p=phl_anls&i=20260203114504936guo1xc48hrs5&token=ejdCZklQclc3WlkydTZGcGl4TFREd2lJTTZvPSxPRFBTX09CTzpwNF8yNjE4MzAxNzM0MTg2MTQ1NDcsMTc3MjcxMTEwNSx7IlN0YXRlbWVudCI6W3siQWN0aW9uIjpbIm9kcHM6UmVhZCJdLCJFZmZlY3QiOiJBbGxvdyIsIlJlc291cmNlIjpbImFjczpvZHBzOio6cHJvamVjdHMvcGhsX2FubHMvaW5zdGFuY2VzLzIwMjYwMjAzMTE0NTA0OTM2Z3VvMXhjNDhocnM1Il19XSwiVmVyc2lvbiI6IjEifQ==


### 1.4 CL mob×过程指标

In [14]:
# mob过程指标
sql=f'''
WITH tmp AS 
(
    SELECT  mob, modelbin_a, modelbin_c
        ,a.*
    FROM   (
                    SELECT  DATEADD(TO_DATE(dt),-overdue_days_dt,'day') AS due_date
                            ,owner_bucket
                            ,user_type
                            ,owner_id
                            ,borrower_id
                            ,global_case_id
                            ,dt
                            ,call_contact_mobile_times 
                            ,call_connect_contact_mobile_times
                            ,call_contact_mobile_billsec
                            ,comm_times
                            ,call_times
                            ,call_connect_times
                            ,call_duration
                            ,call_ring_duration
                            ,call_billsec
                            ,call_mobile_cnt
                            ,call_connect_mobile_cnt
                            ,is_case_comm
                            ,case_status
                            ,trans_time
                            ,sum(call_times) over (partition by dt) as call_times_check
                            -- ,sum(call_times) over (partition by dt, owner_id) as call_times_agent_check
                            ,MAX(HOUR(last_call_time) + MINUTE(last_call_time) / 60) OVER (PARTITION BY dt,owner_id)-MIN(HOUR(first_call_time) + MINUTE(first_call_time) / 60) OVER (PARTITION BY dt,owner_id) as work_hour
                    FROM    phl_data.dws_fact_coll_case_process_info_daily
                    WHERE   dt >= '2024-10-20'
                    AND     TO_CHAR(DATEADD(TO_DATE(dt),-overdue_days_dt,'day'),'yyyymm') BETWEEN '{start_duemon}' AND '{end_duemon}'
                    AND     owner_bucket IN ('S0','S1','S2','M1')
                ) a
    LEFT JOIN   ( --案件债务关联表
                    SELECT  debt_id
                            ,global_case_id
                            ,SUBSTR(starttime,1,10) AS startday
                            ,SUBSTR(endtime,1,10) AS endday
                    FROM    phl_data.dwb_coll_sword_debt_case_info
                ) b
    ON      a.global_case_id = b.global_case_id
    AND     a.dt >= startday 
    AND     a.dt <= endday
    LEFT JOIN    (
            SELECT  user_id
                    ,debt_id
                    ,due_date
                    ,mob
                    ,case when model_bin >='E' then 'E+' else model_bin end as modelbin_a
                    ,flag_collect_bin as modelbin_c
            FROM    phl_anls.tmp_zyt_cl_vintage
            WHERE   TO_CHAR(due_date,'yyyymm')  BETWEEN '{start_duemon}' AND '{end_duemon}'
            and model_bin is not null
        ) v
    ON      v.debt_id = b.debt_id
)


SELECT  mob
        ,owner_bucket

        ,sum(owner_cnt) / count(distinct dt) AS owner_cnt_avg
        ,SUM(owing_case_cnt) / SUM(owner_cnt) AS owing_case_avg
        ,sum(call_connect_times) / sum(owner_cnt) AS call_connect_times_pp_avg
        ,sum(call_case_cnt) / sum(owing_case_cnt) as call_case_pct
        ,'' as call_duration_avg_owner
        ,case when sum(owing_case_cnt)>0 then sum(comm_times)/(sum(owing_case_cnt)-sum(own_uncomm_case_cnt)) end as cover_times
        ,SUM(call_times) / SUM(owing_case_cnt) AS call_times_avg
        ,SUM(call_duration) / SUM(owing_case_cnt) AS call_duration_avg
        ,SUM(call_contact_mobile_times) / sum(owing_case_cnt) as call_contact_mobile_times_avg

        ,SUM(call_connect_times) / SUM(owing_case_cnt) AS case_connect_times_avg
        ,SUM(call_connect_case_cnt) / SUM(call_case_cnt) AS case_connect_rate
        ,SUM(call_connect_times) / SUM(call_times) AS call_connect_rate
        ,sum(call_connect_contact_mobile_times) / sum(call_contact_mobile_times) as contact_call_connect_rate
        ,SUM(connect_duration) / SUM(call_connect_case_cnt) as connect_duration_case_avg
FROM    (
            SELECT  owner_bucket
                    ,mob

                    ,dt
                    ,count(distinct case when work_hour >= 8 then owner_id end) AS owner_cnt
                    ,COUNT(DISTINCT global_case_id) AS case_cnt
                    ,SUM(CASE    WHEN (case_status <> 2 OR SUBSTR(trans_time,1,10) <> dt) THEN 1 ELSE 0 END) AS owing_case_cnt
                    ,SUM(call_times) AS call_times
                    ,sum(comm_times) as comm_times
                    ,SUM(call_duration) / 60 AS call_duration
                    ,SUM(call_connect_times) AS call_connect_times
                    ,SUM(call_billsec) / 60 AS connect_duration
                    ,SUM(call_contact_mobile_times) as call_contact_mobile_times
                    ,SUM(call_connect_contact_mobile_times) as call_connect_contact_mobile_times
                    ,SUM(CASE    WHEN is_case_comm = 0 THEN 1 ELSE 0 END) as own_uncomm_case_cnt
                    ,COUNT(DISTINCT 
                          CASE    WHEN ((call_connect_times) > 0) AND (case_status <> 2 OR SUBSTR(trans_time,1,10) <> dt)
                          THEN global_case_id END
                    ) AS call_connect_case_cnt
                    ,COUNT(DISTINCT CASE    WHEN ((call_times) > 0) AND (case_status <> 2 OR SUBSTR(trans_time,1,10) <> dt) 
                    THEN global_case_id END) AS call_case_cnt
            FROM    tmp
            where call_times_check > 2000
            GROUP BY owner_bucket
                     ,user_type
                     ,dt
                     ,mob
        ) 
GROUP BY owner_bucket, mob
;
    '''
df = run_sql(sql)
df_mob_process = df[df['mob'].notna()].sort_values('mob')
df_mob_process = generate_bucket(df_mob_process, ['S0','S1','S2','M1'])

log_view_address: http://logview.odps.aliyun.com/logview/?h=https://service.ap-southeast-1-vpc.maxcompute.aliyun-inc.com/api&p=phl_anls&i=20260203114522492g5i430u1al7&token=dkM0SU14bkQxT2Q2V3hMTWZUREhNU3lnNEhVPSxPRFBTX09CTzpwNF8yNjE4MzAxNzM0MTg2MTQ1NDcsMTc3MjcxMTEyMix7IlN0YXRlbWVudCI6W3siQWN0aW9uIjpbIm9kcHM6UmVhZCJdLCJFZmZlY3QiOiJBbGxvdyIsIlJlc291cmNlIjpbImFjczpvZHBzOio6cHJvamVjdHMvcGhsX2FubHMvaW5zdGFuY2VzLzIwMjYwMjAzMTE0NTIyNDkyZzVpNDMwdTFhbDciXX1dLCJWZXJzaW9uIjoiMSJ9


In [15]:
df_mob_process

,mob,owner_bucket,owner_cnt_avg,owing_case_avg,call_connect_times_pp_avg,call_case_pct,call_duration_avg_owner,cover_times,call_times_avg,call_duration_avg,...,call_duration_avg_owner,cover_times,call_times_avg,call_duration_avg,call_contact_mobile_times_avg,case_connect_times_avg,case_connect_rate,call_connect_rate,contact_call_connect_rate,connect_duration_case_avg
0,1.0-1,S0,175.270833,9.823517,5.138179,0.853292,,13.191170,12.925861,5.208338,...,,5.692537,4.479772,1.695226,1.640781,0.147103,0.147699,0.032837,0.059553,1.150947
1,2.2-3,S0,88.593750,9.806761,6.078366,0.779466,,14.339210,14.185507,5.765410,...,,6.339592,5.061471,1.869786,1.744143,0.136100,0.136433,0.026890,0.053927,1.147288
2,3.4-5,S0,87.875000,11.267722,7.061522,0.742900,,15.211121,15.013019,6.083746,...,,7.188434,5.838902,2.147308,1.887152,0.136480,0.133439,0.023374,0.049897,1.206712
3,4.6-12,S0,88.046875,34.569476,22.119077,0.592433,,17.255408,17.005323,6.873535,...,,9.072455,7.656516,2.802802,2.261437,0.155577,0.141047,0.020320,0.045906,1.313738
4,5.13-18,S0,88.455497,17.782539,11.203492,0.576509,,18.418693,18.129375,7.297418,...,,10.499356,9.157611,3.331419,2.669705,0.166554,0.140923,0.018187,0.040649,1.430216
5,6.19-24,S0,87.614583,10.066282,6.054274,0.581215,,18.334075,18.079239,7.252960,...,,10.940854,9.580566,3.471578,2.766177,0.161820,0.139453,0.016890,0.039259,1.445812
6,7.24+,S0,87.005208,9.050224,5.701287,0.553802,,18.138117,17.939656,7.085839,...,,10.724073,9.282328,3.260818,2.681954,0.154981,0.137554,0.016696,0.038270,1.428217


In [16]:
# modelbin a过程指标
sql=f'''
WITH tmp AS 
(
    SELECT  mob, modelbin_a, modelbin_c
        ,a.*
    FROM   (
                    SELECT  DATEADD(TO_DATE(dt),-overdue_days_dt,'day') AS due_date
                            ,owner_bucket
                            ,user_type
                            ,owner_id
                            ,borrower_id
                            ,global_case_id
                            ,dt
                            ,call_contact_mobile_times 
                            ,call_connect_contact_mobile_times
                            ,call_contact_mobile_billsec
                            ,comm_times
                            ,call_times
                            ,call_connect_times
                            ,call_duration
                            ,call_ring_duration
                            ,call_billsec
                            ,call_mobile_cnt
                            ,call_connect_mobile_cnt
                            ,is_case_comm
                            ,case_status
                            ,trans_time
                            ,sum(call_times) over (partition by dt) as call_times_check
                            -- ,sum(call_times) over (partition by dt, owner_id) as call_times_agent_check
                            ,MAX(HOUR(last_call_time) + MINUTE(last_call_time) / 60) OVER (PARTITION BY dt,owner_id)-MIN(HOUR(first_call_time) + MINUTE(first_call_time) / 60) OVER (PARTITION BY dt,owner_id) as work_hour
                    FROM    phl_data.dws_fact_coll_case_process_info_daily
                    WHERE   dt >= '2024-10-20'
                    AND     TO_CHAR(DATEADD(TO_DATE(dt),-overdue_days_dt,'day'),'yyyymm') BETWEEN '{start_duemon}' AND '{end_duemon}'
                    AND     owner_bucket IN ('S0','S1','S2','M1')
                ) a
    LEFT JOIN   ( --案件债务关联表
                    SELECT  debt_id
                            ,global_case_id
                            ,SUBSTR(starttime,1,10) AS startday
                            ,SUBSTR(endtime,1,10) AS endday
                    FROM    phl_data.dwb_coll_sword_debt_case_info
                ) b
    ON      a.global_case_id = b.global_case_id
    AND     a.dt >= startday 
    AND     a.dt <= endday
    LEFT JOIN    (
            SELECT  user_id
                    ,debt_id
                    ,due_date
                    ,mob
                    ,case when model_bin >='E' then 'E+' else model_bin end as modelbin_a
                    ,flag_collect_bin as modelbin_c
            FROM    phl_anls.tmp_zyt_cl_vintage
            WHERE   TO_CHAR(due_date,'yyyymm')  BETWEEN '{start_duemon}' AND '{end_duemon}'
            and model_bin is not null
        ) v
    ON      v.debt_id = b.debt_id
)


SELECT modelbin_a
        ,owner_bucket
        ,user_type

        ,sum(owner_cnt) / count(distinct dt) AS owner_cnt_avg
        ,SUM(owing_case_cnt) / SUM(owner_cnt) AS owing_case_avg
        ,sum(call_connect_times) / sum(owner_cnt) AS call_connect_times_pp_avg
        ,sum(call_case_cnt) / sum(owing_case_cnt) as call_case_pct
        ,'' as call_duration_avg_owner
        ,case when sum(owing_case_cnt)>0 then sum(comm_times)/(sum(owing_case_cnt)-sum(own_uncomm_case_cnt)) end as cover_times
        ,SUM(call_times) / SUM(owing_case_cnt) AS call_times_avg
        ,SUM(call_duration) / SUM(owing_case_cnt) AS call_duration_avg
        ,SUM(call_contact_mobile_times) / sum(owing_case_cnt) as call_contact_mobile_times_avg

        ,SUM(call_connect_times) / SUM(owing_case_cnt) AS case_connect_times_avg
        ,SUM(call_connect_case_cnt) / SUM(call_case_cnt) AS case_connect_rate
        ,SUM(call_connect_times) / SUM(call_times) AS call_connect_rate
        ,sum(call_connect_contact_mobile_times) / sum(call_contact_mobile_times) as contact_call_connect_rate
        ,SUM(connect_duration) / SUM(call_connect_case_cnt) as connect_duration_case_avg
FROM    (
            SELECT  owner_bucket
                    ,modelbin_a
                    ,user_type
                    ,dt
                    ,count(distinct case when work_hour >= 8 then owner_id end) AS owner_cnt
                    ,COUNT(DISTINCT global_case_id) AS case_cnt
                    ,SUM(CASE    WHEN (case_status <> 2 OR SUBSTR(trans_time,1,10) <> dt) THEN 1 ELSE 0 END) AS owing_case_cnt
                    ,SUM(call_times) AS call_times
                    ,sum(comm_times) as comm_times
                    ,SUM(call_duration) / 60 AS call_duration
                    ,SUM(call_connect_times) AS call_connect_times
                    ,SUM(call_billsec) / 60 AS connect_duration
                    ,SUM(call_contact_mobile_times) as call_contact_mobile_times
                    ,SUM(call_connect_contact_mobile_times) as call_connect_contact_mobile_times
                    ,SUM(CASE    WHEN is_case_comm = 0 THEN 1 ELSE 0 END) as own_uncomm_case_cnt
                    ,COUNT(DISTINCT 
                          CASE    WHEN ((call_connect_times) > 0) AND (case_status <> 2 OR SUBSTR(trans_time,1,10) <> dt)
                          THEN global_case_id END
                    ) AS call_connect_case_cnt
                    ,COUNT(DISTINCT CASE    WHEN ((call_times) > 0) AND (case_status <> 2 OR SUBSTR(trans_time,1,10) <> dt) 
                    THEN global_case_id END) AS call_case_cnt
            FROM    tmp
            where call_times_check > 2000
            GROUP BY owner_bucket, modelbin_a, user_type
                     ,dt
                     ,owner_id
        ) 
GROUP BY owner_bucket, user_type, modelbin_a
;
    '''
df = run_sql(sql)
df_modelbina_process = df[df['modelbin_a'].notna()].sort_values(['user_type','modelbin_a'])
df_modelbina_process = generate_bucket(df_modelbina_process, ['S0','S1','S2','M1'])

log_view_address: http://logview.odps.aliyun.com/logview/?h=https://service.ap-southeast-1-vpc.maxcompute.aliyun-inc.com/api&p=phl_anls&i=20260203114614590gpscjvsaq2&token=Z1J6Q3BzWlVBcFNSdlVEVHdOcUgveFJMbmc4PSxPRFBTX09CTzpwNF8yNjE4MzAxNzM0MTg2MTQ1NDcsMTc3MjcxMTE3NCx7IlN0YXRlbWVudCI6W3siQWN0aW9uIjpbIm9kcHM6UmVhZCJdLCJFZmZlY3QiOiJBbGxvdyIsIlJlc291cmNlIjpbImFjczpvZHBzOio6cHJvamVjdHMvcGhsX2FubHMvaW5zdGFuY2VzLzIwMjYwMjAzMTE0NjE0NTkwZ3BzY2p2c2FxMiJdfV0sIlZlcnNpb24iOiIxIn0=


In [17]:
# modelbin c过程指标
sql=f'''
WITH tmp AS 
(
    SELECT  mob, modelbin_a, modelbin_c
        ,a.*
    FROM   (
                    SELECT  DATEADD(TO_DATE(dt),-overdue_days_dt,'day') AS due_date
                            ,owner_bucket
                            ,user_type
                            ,owner_id
                            ,borrower_id
                            ,global_case_id
                            ,dt
                            ,call_contact_mobile_times 
                            ,call_connect_contact_mobile_times
                            ,call_contact_mobile_billsec
                            ,comm_times
                            ,call_times
                            ,call_connect_times
                            ,call_duration
                            ,call_ring_duration
                            ,call_billsec
                            ,call_mobile_cnt
                            ,call_connect_mobile_cnt
                            ,is_case_comm
                            ,case_status
                            ,trans_time
                            ,sum(call_times) over (partition by dt) as call_times_check
                            -- ,sum(call_times) over (partition by dt, owner_id) as call_times_agent_check
                            ,MAX(HOUR(last_call_time) + MINUTE(last_call_time) / 60) OVER (PARTITION BY dt,owner_id)-MIN(HOUR(first_call_time) + MINUTE(first_call_time) / 60) OVER (PARTITION BY dt,owner_id) as work_hour
                    FROM    phl_data.dws_fact_coll_case_process_info_daily
                    WHERE   dt >= '2024-10-20'
                    AND     TO_CHAR(DATEADD(TO_DATE(dt),-overdue_days_dt,'day'),'yyyymm') BETWEEN '{start_duemon}' AND '{end_duemon}'
                    AND     owner_bucket IN ('S0','S1','S2','M1')
                ) a
    LEFT JOIN   ( --案件债务关联表
                    SELECT  debt_id
                            ,global_case_id
                            ,SUBSTR(starttime,1,10) AS startday
                            ,SUBSTR(endtime,1,10) AS endday
                    FROM    phl_data.dwb_coll_sword_debt_case_info
                ) b
    ON      a.global_case_id = b.global_case_id
    AND     a.dt >= startday 
    AND     a.dt <= endday
    LEFT JOIN    (
            SELECT  user_id
                    ,debt_id
                    ,due_date
                    ,mob
                    ,case when model_bin >='E' then 'E+' else model_bin end as modelbin_a
                    ,flag_collect_bin as modelbin_c
            FROM    phl_anls.tmp_zyt_cl_vintage
            WHERE   TO_CHAR(due_date,'yyyymm')  BETWEEN '{start_duemon}' AND '{end_duemon}'
            and model_bin is not null
        ) v
    ON      v.debt_id = b.debt_id
)


SELECT  modelbin_c
        ,owner_bucket
        ,user_type

        ,sum(owner_cnt) / count(distinct dt) AS owner_cnt_avg
        ,SUM(owing_case_cnt) / SUM(owner_cnt) AS owing_case_avg
        ,sum(call_connect_times) / sum(owner_cnt) AS call_connect_times_pp_avg
        ,sum(call_case_cnt) / sum(owing_case_cnt) as call_case_pct
        ,'' as call_duration_avg_owner
        ,case when sum(owing_case_cnt)>0 then sum(comm_times)/(sum(owing_case_cnt)-sum(own_uncomm_case_cnt)) end as cover_times
        ,SUM(call_times) / SUM(owing_case_cnt) AS call_times_avg
        ,SUM(call_duration) / SUM(owing_case_cnt) AS call_duration_avg
        ,SUM(call_contact_mobile_times) / sum(owing_case_cnt) as call_contact_mobile_times_avg

        ,SUM(call_connect_times) / SUM(owing_case_cnt) AS case_connect_times_avg
        ,SUM(call_connect_case_cnt) / SUM(call_case_cnt) AS case_connect_rate
        ,SUM(call_connect_times) / SUM(call_times) AS call_connect_rate
        ,sum(call_connect_contact_mobile_times) / sum(call_contact_mobile_times) as contact_call_connect_rate
        ,SUM(connect_duration) / SUM(call_connect_case_cnt) as connect_duration_case_avg
FROM    (
            SELECT  owner_bucket

                    ,modelbin_c
                    ,user_type
                    ,dt
                    ,count(distinct case when work_hour >= 8 then owner_id end) AS owner_cnt
                    ,COUNT(DISTINCT global_case_id) AS case_cnt
                    ,SUM(CASE    WHEN (case_status <> 2 OR SUBSTR(trans_time,1,10) <> dt) THEN 1 ELSE 0 END) AS owing_case_cnt
                    ,SUM(call_times) AS call_times
                    ,sum(comm_times) as comm_times
                    ,SUM(call_duration) / 60 AS call_duration
                    ,SUM(call_connect_times) AS call_connect_times
                    ,SUM(call_billsec) / 60 AS connect_duration
                    ,SUM(call_contact_mobile_times) as call_contact_mobile_times
                    ,SUM(call_connect_contact_mobile_times) as call_connect_contact_mobile_times
                    ,SUM(CASE    WHEN is_case_comm = 0 THEN 1 ELSE 0 END) as own_uncomm_case_cnt
                    ,COUNT(DISTINCT 
                          CASE    WHEN ((call_connect_times) > 0) AND (case_status <> 2 OR SUBSTR(trans_time,1,10) <> dt)
                          THEN global_case_id END
                    ) AS call_connect_case_cnt
                    ,COUNT(DISTINCT CASE    WHEN ((call_times) > 0) AND (case_status <> 2 OR SUBSTR(trans_time,1,10) <> dt) 
                    THEN global_case_id END) AS call_case_cnt
            FROM    tmp
            WHERE   call_times_check > 2000
            GROUP BY owner_bucket, user_type, modelbin_c
                     ,dt
                     ,owner_id
        ) 
GROUP BY owner_bucket, user_type, modelbin_c
;
    '''
df = run_sql(sql)
df_modelbinc_process = df[(df['modelbin_c'].notna()) & (df['modelbin_c']!='err')].sort_values(['user_type','modelbin_c'])
df_modelbinc_process = generate_bucket(df_modelbinc_process, ['S0','S1','S2','M1'])

log_view_address: http://logview.odps.aliyun.com/logview/?h=https://service.ap-southeast-1-vpc.maxcompute.aliyun-inc.com/api&p=phl_anls&i=20260203114717988ggcpy851zw6&token=QTNjaFZpeXlhZ0hUNm1zWE1nTHNaOWduNi9BPSxPRFBTX09CTzpwNF8yNjE4MzAxNzM0MTg2MTQ1NDcsMTc3MjcxMTIzOCx7IlN0YXRlbWVudCI6W3siQWN0aW9uIjpbIm9kcHM6UmVhZCJdLCJFZmZlY3QiOiJBbGxvdyIsIlJlc291cmNlIjpbImFjczpvZHBzOio6cHJvamVjdHMvcGhsX2FubHMvaW5zdGFuY2VzLzIwMjYwMjAzMTE0NzE3OTg4Z2djcHk4NTF6dzYiXX1dLCJWZXJzaW9uIjoiMSJ9


In [18]:
# period_no过程指标
sql=f'''
WITH tmp AS 
(
    SELECT  mob, modelbin_a, modelbin_c, period_no
        ,a.*
    FROM   (
                    SELECT  DATEADD(TO_DATE(dt),-overdue_days_dt,'day') AS due_date
                            ,owner_bucket
                            ,user_type
                            ,owner_id
                            ,borrower_id
                            ,global_case_id
                            ,dt
                            ,call_contact_mobile_times 
                            ,call_connect_contact_mobile_times
                            ,call_contact_mobile_billsec
                            ,comm_times
                            ,call_times
                            ,call_connect_times
                            ,call_duration
                            ,call_ring_duration
                            ,call_billsec
                            ,call_mobile_cnt
                            ,call_connect_mobile_cnt
                            ,is_case_comm
                            ,case_status
                            ,trans_time
                            ,sum(call_times) over (partition by dt) as call_times_check
                            -- ,sum(call_times) over (partition by dt, owner_id) as call_times_agent_check
                            ,MAX(HOUR(last_call_time) + MINUTE(last_call_time) / 60) OVER (PARTITION BY dt,owner_id)-MIN(HOUR(first_call_time) + MINUTE(first_call_time) / 60) OVER (PARTITION BY dt,owner_id) as work_hour
                    FROM    phl_data.dws_fact_coll_case_process_info_daily
                    WHERE   dt >= '2024-10-20'
                    AND     TO_CHAR(DATEADD(TO_DATE(dt),-overdue_days_dt,'day'),'yyyymm') BETWEEN '{start_duemon}' AND '{end_duemon}'
                    AND     owner_bucket IN ('S0','S1','S2','M1')
                ) a
    LEFT JOIN   ( --案件债务关联表
                    SELECT  debt_id
                            ,global_case_id
                            ,SUBSTR(starttime,1,10) AS startday
                            ,SUBSTR(endtime,1,10) AS endday
                    FROM    phl_data.dwb_coll_sword_debt_case_info
                ) b
    ON      a.global_case_id = b.global_case_id
    AND     a.dt >= startday 
    AND     a.dt <= endday
    LEFT JOIN    (
            SELECT  user_id
                    ,debt_id
                    ,due_date
                    ,mob
                    ,period_no
                    ,case when model_bin >='E' then 'E+' else model_bin end as modelbin_a
                    ,flag_collect_bin as modelbin_c
            FROM    phl_anls.tmp_zyt_cl_vintage
            WHERE   TO_CHAR(due_date,'yyyymm')  BETWEEN '{start_duemon}' AND '{end_duemon}'
            and model_bin is not null
        ) v
    ON      v.debt_id = b.debt_id
)


SELECT  period_no
        ,owner_bucket
        ,user_type

        ,sum(owner_cnt) / count(distinct dt) AS owner_cnt_avg
        ,SUM(owing_case_cnt) / SUM(owner_cnt) AS owing_case_avg
        ,sum(call_connect_times) / sum(owner_cnt) AS call_connect_times_pp_avg
        ,sum(call_case_cnt) / sum(owing_case_cnt) as call_case_pct
        ,'' as call_duration_avg_owner
        ,case when sum(owing_case_cnt)>0 then sum(comm_times)/(sum(owing_case_cnt)-sum(own_uncomm_case_cnt)) end as cover_times
        ,SUM(call_times) / SUM(owing_case_cnt) AS call_times_avg
        ,SUM(call_duration) / SUM(owing_case_cnt) AS call_duration_avg
        ,SUM(call_contact_mobile_times) / sum(owing_case_cnt) as call_contact_mobile_times_avg

        ,SUM(call_connect_times) / SUM(owing_case_cnt) AS case_connect_times_avg
        ,SUM(call_connect_case_cnt) / SUM(call_case_cnt) AS case_connect_rate
        ,SUM(call_connect_times) / SUM(call_times) AS call_connect_rate
        ,sum(call_connect_contact_mobile_times) / sum(call_contact_mobile_times) as contact_call_connect_rate
        ,SUM(connect_duration) / SUM(call_connect_case_cnt) as connect_duration_case_avg
FROM    (
            SELECT  owner_bucket

                    ,period_no
                    ,user_type
                    ,dt
                    ,count(distinct case when work_hour >= 8 then owner_id end) AS owner_cnt
                    ,COUNT(DISTINCT global_case_id) AS case_cnt
                    ,SUM(CASE    WHEN (case_status <> 2 OR SUBSTR(trans_time,1,10) <> dt) THEN 1 ELSE 0 END) AS owing_case_cnt
                    ,SUM(call_times) AS call_times
                    ,sum(comm_times) as comm_times
                    ,SUM(call_duration) / 60 AS call_duration
                    ,SUM(call_connect_times) AS call_connect_times
                    ,SUM(call_billsec) / 60 AS connect_duration
                    ,SUM(call_contact_mobile_times) as call_contact_mobile_times
                    ,SUM(call_connect_contact_mobile_times) as call_connect_contact_mobile_times
                    ,SUM(CASE    WHEN is_case_comm = 0 THEN 1 ELSE 0 END) as own_uncomm_case_cnt
                    ,COUNT(DISTINCT 
                          CASE    WHEN ((call_connect_times) > 0) AND (case_status <> 2 OR SUBSTR(trans_time,1,10) <> dt)
                          THEN global_case_id END
                    ) AS call_connect_case_cnt
                    ,COUNT(DISTINCT CASE    WHEN ((call_times) > 0) AND (case_status <> 2 OR SUBSTR(trans_time,1,10) <> dt) 
                    THEN global_case_id END) AS call_case_cnt
            FROM    tmp
            where call_times_check > 2000
            GROUP BY owner_bucket, user_type, period_no
                     ,dt
                     ,owner_id
        ) 
GROUP BY owner_bucket, user_type, period_no
;
    '''
df = run_sql(sql)
df_period_process = df[(df['period_no'].notna()) & (df['period_no']!='err')].sort_values(['user_type','period_no'])
df_period_process = generate_bucket(df_period_process, ['S0','S1','S2','M1'])

log_view_address: http://logview.odps.aliyun.com/logview/?h=https://service.ap-southeast-1-vpc.maxcompute.aliyun-inc.com/api&p=phl_anls&i=20260203114809692gpjukot1al7&token=MGJjL1ZQWnVnVWtzM0w1Qi9SZ3M4V2doR3N3PSxPRFBTX09CTzpwNF8yNjE4MzAxNzM0MTg2MTQ1NDcsMTc3MjcxMTI4OSx7IlN0YXRlbWVudCI6W3siQWN0aW9uIjpbIm9kcHM6UmVhZCJdLCJFZmZlY3QiOiJBbGxvdyIsIlJlc291cmNlIjpbImFjczpvZHBzOio6cHJvamVjdHMvcGhsX2FubHMvaW5zdGFuY2VzLzIwMjYwMjAzMTE0ODA5NjkyZ3BqdWtvdDFhbDciXX1dLCJWZXJzaW9uIjoiMSJ9


### 1.TOTAL

In [19]:
# with pd.ExcelWriter('output.xlsx') as writer:
#     df_duemon.to_excel(writer, sheet_name='Cashloan', startrow=2, startcol=0, index=True)
#     df_duemon_process.to_excel(writer, sheet_name='Cashloan', startrow=2, startcol=30, index=True)
    
#     df_principal.to_excel(writer, sheet_name='Cashloan', startrow=6+df_duemon.shape[0], startcol=0, index=True)
#     df_principal_process.to_excel(writer, sheet_name='Cashloan', startrow=6+df_duemon.shape[0], startcol=30, index=True)
#     df_principal_duemon_result.to_excel(writer, sheet_name='Cashloan', startrow=10+df_duemon.shape[0]*2, startcol=0, index=True)
    
#     df_mob.to_excel(writer, sheet_name='Cashloan', startrow=14+df_duemon.shape[0]*3, startcol=0, index=True)
#     df_mob_process.to_excel(writer, sheet_name='Cashloan', startrow=14+df_duemon.shape[0]*3, startcol=30, index=True)
    
#     df_modelbina.to_excel(writer, sheet_name='Cashloan', startrow=18+df_duemon.shape[0]*4, startcol=0, index=True)
#     df_modelbina_process.to_excel(writer, sheet_name='Cashloan', startrow=18+df_duemon.shape[0]*4, startcol=30, index=True)
#     df_modelbina_duemon_result.to_excel(writer, sheet_name='Cashloan', startrow=22+df_duemon.shape[0]*5, startcol=0, index=True)
    
#     df_modelbinc.to_excel(writer, sheet_name='Cashloan', startrow=26+df_duemon.shape[0]*6, startcol=0, index=True)
#     df_modelbinc_process.to_excel(writer, sheet_name='Cashloan', startrow=26+df_duemon.shape[0]*6, startcol=30, index=True)
#     df_modelbinc_duemon_result.to_excel(writer, sheet_name='Cashloan', startrow=30+df_duemon.shape[0]*7, startcol=0, index=True)
    
#     df_period.to_excel(writer, sheet_name='Cashloan', startrow=34+df_duemon.shape[0]*8, startcol=0, index=True)
#     df_period_process.to_excel(writer, sheet_name='Cashloan', startrow=34+df_duemon.shape[0]*8, startcol=30, index=True)
#     df_period_duemon_result.to_excel(writer, sheet_name='Cashloan', startrow=38+df_duemon.shape[0]*9, startcol=30, index=True)

In [36]:
with pd.ExcelWriter('output.xlsx') as writer:  
    df_modelbinc.to_excel(writer, sheet_name='Cashloan', startrow=26+df_duemon.shape[0]*6, startcol=0, index=True)
    df_modelbinc_process.to_excel(writer, sheet_name='Cashloan', startrow=26+df_duemon.shape[0]*6, startcol=30, index=True)
    df_modelbinc_duemon_result.to_excel(writer, sheet_name='Cashloan', startrow=30+df_duemon.shape[0]*7, startcol=0, index=True)
    

In [37]:
df_total_cl

,user_type,period_no,modelbin_a,owing_user_cnt,owing_principal,user_cnt_pct,owing_principal_avg,period_seq_avg,overdue_rate,dpd1_repay,...,cover_times,call_times_avg,call_duration_avg,call_contact_mobile_times_avg,case_connect_times_avg,case_connect_rate,call_connect_rate,contact_call_connect_rate,connect_duration_case_avg,tag
0,新客,NaN,None,292073,1045206460,,3578.579533198892057807,1.205373,0.161832728789295849,0.113325690382127795,...,5.609641,4.298958,1.662558,1.564771,0.160125,0.162071,0.037247,0.063372,1.137229,Cashloan
1,老客,NaN,None,6144170,13878344458,,2258.782627759323065605,3.163957,0.083246659419382456,0.355383439556598208,...,7.709454,6.281897,2.308964,1.983987,0.144290,0.138552,0.022969,0.049119,1.147013,Cashloan
2,None,NaN,None,6436243,14923550918,,2318.674251112022961221,3.075229,0.088750622103784214,0.324470247265305531,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Cashloan


# TT

In [38]:
sql = f'''
SELECT  case when user_type = '新客' then '新客'
            when user_type in ('新转化老客','存量老客') then '老客'
            else 'total' end as user_type
        ,TO_CHAR(due_date,'yyyymm') AS due_mon
        ,flag_tt_principal
        ,case when modelbinnew >='E' then 'E+' else modelbinnew end as modelbin_a
        ,case when s0_bucket in ('B01','B02') then 'E'
            when s0_bucket in ('B03','B04','B05') then 'D'
            when s0_bucket in ('B06','B07','B08','B09') then 'C'
            when s0_bucket in ('B10','B11','B12','B13') then 'B'
            else 'A'
            end as modelbin_c
        ,mob
        ,SUM(owing_user_cnt) as owing_user_cnt
        ,SUM(owing_principal) as owing_principal
        ,'' as user_cnt_pct
        ,sum(owing_principal)/sum(owing_user_cnt) as owing_principal_avg
        ,AVG(period_no_tt) as period_no_avg
        ,SUM(case when datediff(now(),to_date(due_date))>=1 then overdue_principal end)/ SUM(case when datediff(now(),to_date(due_date))>=1 then owing_principal end) AS overdue_rate
        ,case when SUM(case when datediff(now(),to_date(due_date))>=2 then overdue_principal end) = 0 then '/' else 1 - SUM(case when datediff(now(),to_date(due_date))>=2 then d2_principal end)/ SUM(case when datediff(now(),to_date(due_date))>=2 then overdue_principal end) end AS dpd1_repay
        ,case when SUM(case when datediff(now(),to_date(due_date))>=6 then overdue_principal end) = 0 then '/' else 1 - SUM(case when datediff(now(),to_date(due_date))>=6 then d6_principal end)/ SUM(case when datediff(now(),to_date(due_date))>=6 then overdue_principal end) end AS dpd5_repay
        ,case when SUM(case when datediff(now(),to_date(due_date))>=8 then overdue_principal end) = 0 then '/' else 1 - SUM(case when datediff(now(),to_date(due_date))>=8 then d8_principal end)/ SUM(case when datediff(now(),to_date(due_date))>=8 then overdue_principal end) end AS dpd7_repay
        ,case when SUM(case when datediff(now(),to_date(due_date))>=16 then overdue_principal end) = 0 then '/' else 1 - SUM(case when datediff(now(),to_date(due_date))>=16 then d16_principal end)/ SUM(case when datediff(now(),to_date(due_date))>=16 then overdue_principal end) end AS dpd15_repay
        ,case when SUM(case when datediff(now(),to_date(due_date))>=31 then overdue_principal end) = 0 then '/' else 1 - SUM(case when datediff(now(),to_date(due_date))>=31 then d31_principal end)/ SUM(case when datediff(now(),to_date(due_date))>=31 then overdue_principal end) end AS dpd30_repay
        ,SUM(case when datediff(now(),to_date(due_date))>=6 then d6_principal end)/ SUM(case when datediff(now(),to_date(due_date))>=6 then owing_principal end) AS dpd5
        ,case when SUM(case when datediff(now(),to_date(due_date))>=31 then owing_principal end) = 0 then '/' else SUM(case when datediff(now(),to_date(due_date))>=31 then d31_principal end)/ SUM(case when datediff(now(),to_date(due_date))>=31 then owing_principal end) end AS dpd30
        ,case when sum(owing_user_cnt) = 0 then '/' else sum(case when is_touch0=2 then owing_user_cnt end)/sum(owing_user_cnt) end as connect_rate0
        ,case when sum(owing_user_cnt) = 0 then '/' else sum(case when action_code0 like '%PTP' then owing_user_cnt end)/sum(owing_user_cnt) end as ptp_rate0
        ,case when sum(case when is_touch0=2 then overdue_principal end)=0 then '/' else sum(case when is_touch0=2 then overdue_principal-d11_principal end)/sum(case when is_touch0=2 then overdue_principal end) end as connect_conversion0
        ,case when sum(case when action_code0 like '%PTP' then overdue_principal end)=0 then '/' else sum(case when action_code0 like '%PTP' then overdue_principal-d11_principal end)/sum(case when action_code0 like '%PTP' then overdue_principal end) end as ptp_conversion0
        ,case when sum(case when is_touch0=1 then overdue_principal end)=0 then '/' else sum(case when is_touch0=1 then overdue_principal-d11_principal end)/sum(case when is_touch0=1 then overdue_principal end) end as disconnect_conversion0

        ,case when sum(case when is_touch is not null then owing_user_cnt end)=0 then '/' else sum(case when is_touch=2 then owing_user_cnt end)/sum(case when is_touch is not null then owing_user_cnt end) end as connect_rate
        ,case when sum(case when is_touch is not null then owing_user_cnt end)=0 then '/' else sum(case when action_code like '%PTP' then owing_user_cnt end)/sum(case when is_touch is not null then owing_user_cnt end) end as ptp_rate
        ,case when sum(case when is_touch=2 then overdue_principal end)=0 then '/' else sum(case when is_touch=2 then overdue_principal-d11_principal end)/sum(case when is_touch=2 then overdue_principal end) end as connect_conversion
        ,case when sum(case when action_code like '%PTP' then overdue_principal end)=0 then '/' else sum(case when action_code like '%PTP' then overdue_principal-d11_principal end)/sum(case when action_code like '%PTP' then overdue_principal end) end as ptp_conversion
        ,case when sum(case when is_touch=1 then overdue_principal end)=0 then '/' else sum(case when is_touch=1 then overdue_principal-d11_principal end)/sum(case when is_touch=1 then overdue_principal end) end as disconnect_conversion
FROM    phl_anls.tmp_yp_phl_ana_09_eoc_tt_sum_daily_temp
WHERE to_char(due_date,'yyyymm') BETWEEN '{start_duemon}' AND '{end_duemon}'

GROUP BY 
GROUPING SETS (()
              ,(case when user_type = '新客' then '新客'
            when user_type in ('新转化老客','存量老客') then '老客'
            else 'total' end)
              ,(case when user_type = '新客' then '新客'
            when user_type in ('新转化老客','存量老客') then '老客'
            else 'total' end,TO_CHAR(due_date,'yyyymm'))
            ,(case when user_type = '新客' then '新客'
            when user_type in ('新转化老客','存量老客') then '老客'
            else 'total' end, case when modelbinnew >='E' then 'E+' else modelbinnew end
                )
            ,(case when user_type = '新客' then '新客'
            when user_type in ('新转化老客','存量老客') then '老客'
            else 'total' end 
             ,case when s0_bucket in ('B01','B02') then 'E'
            when s0_bucket in ('B03','B04','B05') then 'D'
            when s0_bucket in ('B06','B07','B08','B09') then 'C'
            when s0_bucket in ('B10','B11','B12','B13') then 'B'
            else 'A'
            end)
              ,(case when user_type = '新客' then '新客'
            when user_type in ('新转化老客','存量老客') then '老客'
            else 'total' end,flag_tt_principal)
            ,(mob)
            ,(case when user_type = '新客' then '新客'
            when user_type in ('新转化老客','存量老客') then '老客'
            else 'total' end,flag_tt_principal,to_char(due_date,'yyyymm'))
            ,(case when user_type = '新客' then '新客'
            when user_type in ('新转化老客','存量老客') then '老客'
            else 'total' end,case when modelbinnew >='E' then 'E+' else modelbinnew end,to_char(due_date,'yyyymm'))
            ,(case when user_type = '新客' then '新客'
            when user_type in ('新转化老客','存量老客') then '老客'
            else 'total' end,case when s0_bucket in ('B01','B02') then 'E'
            when s0_bucket in ('B03','B04','B05') then 'D'
            when s0_bucket in ('B06','B07','B08','B09') then 'C'
            when s0_bucket in ('B10','B11','B12','B13') then 'B'
            else 'A'
            end,to_char(due_date,'yyyymm'))
)
'''
df = run_sql(sql)

log_view_address: http://logview.odps.aliyun.com/logview/?h=https://service.ap-southeast-1-vpc.maxcompute.aliyun-inc.com/api&p=phl_anls&i=2026020311584980gwwsy851zw6&token=cnBCNUdHQ01OVXhQQktwSlpnMCtvTEt4RGswPSxPRFBTX09CTzpwNF8yNjE4MzAxNzM0MTg2MTQ1NDcsMTc3MjcxMTkyOSx7IlN0YXRlbWVudCI6W3siQWN0aW9uIjpbIm9kcHM6UmVhZCJdLCJFZmZlY3QiOiJBbGxvdyIsIlJlc291cmNlIjpbImFjczpvZHBzOio6cHJvamVjdHMvcGhsX2FubHMvaW5zdGFuY2VzLzIwMjYwMjAzMTE1ODQ5ODBnd3dzeTg1MXp3NiJdfV0sIlZlcnNpb24iOiIxIn0=


In [39]:
df_total_tt = df[(df['flag_tt_principal'].isna()) & (df['due_mon'].isna()) & (df['mob'].isna()) & (df['modelbin_a'].isna()) & (df['modelbin_c'].isna())].drop(columns=['due_mon','flag_tt_principal','mob']).sort_values(by='user_type')
df_duemon = df[df['due_mon'].notna()&df['flag_tt_principal'].isna()&df['modelbin_a'].isna()&df['modelbin_c'].isna()].drop(columns=['flag_tt_principal','mob']).sort_values(by=['user_type', 'due_mon'],ascending=[True,False])
df_principal = df[df['flag_tt_principal'].notna()&df['due_mon'].isna()].drop(columns=['due_mon','mob']).sort_values(by=['user_type', 'flag_tt_principal'])
df_mob = df[df['mob'].notna()].drop(columns=['flag_tt_principal','due_mon','user_type']).sort_values(by='mob')
df_modelbina = df[df['modelbin_a'].notna()&df['due_mon'].isna()].drop(columns=['flag_tt_principal','due_mon']).sort_values(by=['user_type','modelbin_a'])
df_modelbinc = df[df['modelbin_c'].notna()&df['due_mon'].isna()].drop(columns=['flag_tt_principal','due_mon']).sort_values(by=['user_type','modelbin_c'])

df_principal_duemon = df[df['flag_tt_principal'].notna() & df['due_mon'].notna()].drop(columns='mob').groupby(by='user_type')
df_principal_duemon_user = df_principal_duemon.apply(lambda x: pd.pivot_table(x,index='flag_tt_principal',columns='due_mon',values='owing_user_cnt',aggfunc='sum'))
df_principal_duemon_user = df_principal_duemon_user.apply(lambda x:x.div(x.sum()))
df_principal_duemon_overdue = df_principal_duemon.apply(lambda x: pd.pivot_table(x,index='flag_tt_principal',columns='due_mon',values='overdue_rate',aggfunc='sum'))
df_principal_duemon_dpd5 = df_principal_duemon.apply(lambda x: pd.pivot_table(x,index='flag_tt_principal',columns='due_mon',values='dpd5',aggfunc='sum'))
df_principal_duemon_result = pd.concat([df_principal_duemon_user,df_principal_duemon_overdue,df_principal_duemon_dpd5],axis=1)

df_modelbina_duemon = df[(df['modelbin_a'].notna()) & (df['due_mon'].notna())].drop(columns='mob').sort_values('modelbin_a').groupby(by='user_type')
df_modelbina_duemon_user = df_modelbina_duemon.apply(lambda x: pd.pivot_table(x, index='modelbin_a', columns='due_mon', values='owing_user_cnt', aggfunc='sum'))
df_modelbina_duemon_user = df_modelbina_duemon_user.apply(lambda x: x.div(x.sum()))
df_modelbina_duemon_overdue = df_modelbina_duemon.apply(lambda x: pd.pivot_table(x,index='modelbin_a',columns='due_mon',values='overdue_rate',aggfunc='sum'))
df_modelbina_duemon_dpd5 = df_modelbina_duemon.apply(lambda x: pd.pivot_table(x,index='modelbin_a',columns='due_mon',values='dpd5',aggfunc='sum'))
df_modelbina_duemon_result = pd.concat([df_modelbina_duemon_user,df_modelbina_duemon_overdue,df_modelbina_duemon_dpd5],axis=1)

df_modelbinc_duemon = df[df['modelbin_c'].notna() & df['due_mon'].notna()].groupby(by='user_type')
df_modelbinc_duemon_user = df_modelbinc_duemon.apply(lambda x: pd.pivot_table(x,index='modelbin_c',columns='due_mon',values='owing_user_cnt',aggfunc='sum'))
df_modelbinc_duemon_user = df_modelbinc_duemon_user.apply(lambda x:x.div(x.sum()))
df_modelbinc_duemon_overdue = df_modelbinc_duemon.apply(lambda x: pd.pivot_table(x,index='modelbin_c',columns='due_mon',values='overdue_rate',aggfunc='sum'))
df_modelbinc_duemon_dpd5 = df_modelbinc_duemon.apply(lambda x: pd.pivot_table(x,index='modelbin_c',columns='due_mon',values='dpd5',aggfunc='sum'))
df_modelbinc_duemon_result = pd.concat([df_modelbinc_duemon_user,df_modelbinc_duemon_overdue,df_modelbinc_duemon_dpd5],axis=1)

### 2.2+3 TT过程指标

In [40]:
# !!NEW!!
# user_type
sql = f'''
    SELECT  owner_bucket
            ,user_type
            
            
            ,sum(owner_cnt) / count(distinct dt) AS owner_cnt_avg
            ,SUM(owing_case_cnt) / SUM(owner_cnt) AS owing_case_avg
            ,sum(call_connect_times) / sum(owner_cnt) AS call_connect_times_pp_avg
            ,sum(call_case_cnt) / sum(owing_case_cnt) as call_case_pct
            ,'' as call_duration_avg_owner
            ,case when sum(owing_case_cnt)>0 then sum(comm_times)/(sum(owing_case_cnt)-sum(own_uncomm_case_cnt)) end as cover_times
            ,SUM(call_times) / SUM(owing_case_cnt) AS call_times_avg
            ,SUM(call_duration) / SUM(owing_case_cnt) AS call_duration_avg
            ,SUM(call_contact_mobile_times) / sum(owing_case_cnt) as call_contact_mobile_times_avg
            
            ,SUM(call_connect_times) / SUM(owing_case_cnt) AS case_connect_times_avg
            ,SUM(call_connect_case_cnt) / SUM(call_case_cnt) AS case_connect_rate
            ,SUM(call_connect_times) / SUM(call_times) AS call_connect_rate
            ,sum(call_connect_contact_mobile_times) / sum(call_contact_mobile_times) as contact_call_connect_rate
            ,SUM(connect_duration) / SUM(call_connect_case_cnt) as connect_duration_case_avg
    FROM    (
                SELECT  case when case_type = 0 then 'T0'
                            when case_type = 1 AND overdue_days_dt <= 7 then 'T1'
                            when case_type = 1 AND overdue_days_dt between 8 and 15 then 'T2'
                            when case_type = 1 AND overdue_days_dt between 16 and 30 then 'TM1'
                            END AS owner_bucket
                        ,user_type
                        ,dt
                        ,datediff(CAST(max(last_comm_time) AS DATETIME), CAST(min(first_comm_time) AS DATETIME), 'hour') AS comm_hour
                        ,count(distinct case when work_hour >= 8 then owner_name end) AS owner_cnt
                        ,COUNT(DISTINCT global_case_id) AS case_cnt
                        ,SUM(CASE    WHEN (case_status <> 2 OR SUBSTR(trans_time,1,10) <> dt) THEN 1 ELSE 0 END) AS owing_case_cnt
                        ,SUM(call_times) AS call_times
                        ,sum(comm_times) as comm_times
                        ,SUM(call_duration) / 60 AS call_duration
                        ,SUM(call_connect_times) AS call_connect_times
                        ,SUM(call_billsec) / 60 AS connect_duration
                        ,SUM(call_contact_mobile_times) as call_contact_mobile_times
                        ,SUM(call_connect_contact_mobile_times) as call_connect_contact_mobile_times
                        ,SUM(CASE    WHEN is_case_comm = 0 THEN 1 ELSE 0 END) as own_uncomm_case_cnt
                        ,COUNT(DISTINCT 
                              CASE    WHEN ((call_connect_times) > 0) AND (case_status <> 2 OR SUBSTR(trans_time,1,10) <> dt)
                              THEN global_case_id END
                        ) AS call_connect_case_cnt
                        ,COUNT(DISTINCT CASE    WHEN ((call_times) > 0) AND (case_status <> 2 OR SUBSTR(trans_time,1,10) <> dt) 
                        THEN global_case_id END) AS call_case_cnt
                FROM    (select *
                        ,sum(call_times) over (partition by dt) as call_times_check
                        -- ,sum(call_times) over (partition by dt, owner_id) as call_times_agent_check
                        ,MAX(HOUR(last_call_time) + MINUTE(last_call_time) / 60) OVER (PARTITION BY dt,owner_id)-MIN(HOUR(first_call_time) + MINUTE(first_call_time) / 60) OVER (PARTITION BY dt,owner_id) as work_hour
                        from phl_data.dws_fact_coll_case_process_info_daily
                        WHERE   dt >= '2024-10-20'
                        AND     TO_CHAR(DATEADD(TO_DATE(dt),-overdue_days_dt,'day'),'yyyymm') BETWEEN '{start_duemon}' AND '{end_duemon}'
                        AND     owner_bucket IN ('T2','T4')
                        )
                where call_times_check > 2000
                GROUP BY case when case_type = 0 then 'T0'
                            when case_type = 1 AND overdue_days_dt <= 7 then 'T1'
                            when case_type = 1 AND overdue_days_dt between 8 and 15 then 'T2'
                            when case_type = 1 AND overdue_days_dt between 16 and 30 then 'TM1'
                            END
                         ,user_type
                         ,dt
                         ,owner_id

            ) a
    GROUP BY owner_bucket
             ,user_type
    ;
'''
df = run_sql(sql)
df_total_tt_process = generate_bucket(df.sort_values('user_type'),['T0','T1','T2','TM1'])
# merged = pd.merge(df_total_cl_process, df, on=['user_type','owner_bucket'], how='left')
# merged = merged.set_index(df_total_cl_process.index)
# df_total_cl_process = generate_bucket(df_total_cl_process, ['S0','S1','S2','M1'])

df_total_tt = pd.concat([df_total_tt.reset_index(drop=True),df_total_tt_process],axis=1)
df_total_tt['tag'] = 'Tiktok'

log_view_address: http://logview.odps.aliyun.com/logview/?h=https://service.ap-southeast-1-vpc.maxcompute.aliyun-inc.com/api&p=phl_anls&i=20260203115905938g0f6xc48hrs5&token=Uk5hR1R5UDcxbHVHZXdLZ3dGeW5HMUNWWGhZPSxPRFBTX09CTzpwNF8yNjE4MzAxNzM0MTg2MTQ1NDcsMTc3MjcxMTk0Nix7IlN0YXRlbWVudCI6W3siQWN0aW9uIjpbIm9kcHM6UmVhZCJdLCJFZmZlY3QiOiJBbGxvdyIsIlJlc291cmNlIjpbImFjczpvZHBzOio6cHJvamVjdHMvcGhsX2FubHMvaW5zdGFuY2VzLzIwMjYwMjAzMTE1OTA1OTM4ZzBmNnhjNDhocnM1Il19XSwiVmVyc2lvbiI6IjEifQ==


In [41]:
df_total_tt

,user_type,modelbin_a,modelbin_c,owing_user_cnt,owing_principal,user_cnt_pct,owing_principal_avg,period_no_avg,overdue_rate,dpd1_repay,...,cover_times,call_times_avg,call_duration_avg,call_contact_mobile_times_avg,case_connect_times_avg,case_connect_rate,call_connect_rate,contact_call_connect_rate,connect_duration_case_avg,tag
0,新客,None,None,1215218,565579318.66,,465.41387525530398661,3.455129,0.182023485165460912,0.221233149117137453,...,3.351541,1.799764,0.680761,0.522468,0.068835,0.110120,0.038247,0.060564,1.075702,Tiktok
1,老客,None,None,12729779,5038592157.71,,395.811440065848747256,4.316972,0.104572650099045002,0.328082866325066419,...,4.548129,2.597609,0.958740,0.771248,0.077861,0.114489,0.029974,0.051713,1.110024,Tiktok
2,None,None,None,13944997,5604171476.37,,401.876850627504616889,4.237183,0.11238907590100586,0.310618259729452911,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Tiktok


In [42]:
# !!NEW!!
# user_type+due_mon
sql = f'''
    SELECT  owner_bucket
            ,user_type
            ,due_mon
            
            ,sum(owner_cnt) / count(distinct dt) AS owner_cnt_avg
            ,SUM(owing_case_cnt) / SUM(owner_cnt) AS owing_case_avg
            ,sum(call_connect_times) / sum(owner_cnt) AS call_connect_times_pp_avg
            ,sum(call_case_cnt) / sum(owing_case_cnt) as call_case_pct
            ,'' as call_duration_avg_owner
            ,case when sum(owing_case_cnt)>0 then sum(comm_times)/(sum(owing_case_cnt)-sum(own_uncomm_case_cnt)) end as cover_times
            ,SUM(call_times) / SUM(owing_case_cnt) AS call_times_avg
            ,SUM(call_duration) / SUM(owing_case_cnt) AS call_duration_avg
            ,SUM(call_contact_mobile_times) / sum(owing_case_cnt) as call_contact_mobile_times_avg
            
            ,SUM(call_connect_times) / SUM(owing_case_cnt) AS case_connect_times_avg
            ,SUM(call_connect_case_cnt) / SUM(call_case_cnt) AS case_connect_rate
            ,SUM(call_connect_times) / SUM(call_times) AS call_connect_rate
            ,sum(call_connect_contact_mobile_times) / sum(call_contact_mobile_times) as contact_call_connect_rate
            ,SUM(connect_duration) / SUM(call_connect_case_cnt) as connect_duration_case_avg
    FROM    (
                SELECT  case when case_type = 0 then 'T0'
                            when case_type = 1 AND overdue_days_dt <= 7 then 'T1'
                            when case_type = 1 AND overdue_days_dt between 8 and 15 then 'T2'
                            when case_type = 1 AND overdue_days_dt between 16 and 30 then 'TM1'
                            END AS owner_bucket
                        ,user_type
                        ,dt
                        ,TO_CHAR(DATEADD(TO_DATE(dt),-overdue_days_dt,'day'),'yyyymm') as due_mon
                        ,datediff(CAST(max(last_comm_time) AS DATETIME), CAST(min(first_comm_time) AS DATETIME), 'hour') AS comm_hour
                        ,count(distinct case when work_hour >= 8 then owner_name end) AS owner_cnt
                        ,COUNT(DISTINCT global_case_id) AS case_cnt
                        ,SUM(CASE    WHEN (case_status <> 2 OR SUBSTR(trans_time,1,10) <> dt) THEN 1 ELSE 0 END) AS owing_case_cnt
                        ,SUM(call_times) AS call_times
                        ,sum(comm_times) as comm_times
                        ,SUM(call_duration) / 60 AS call_duration
                        ,SUM(call_connect_times) AS call_connect_times
                        ,SUM(call_billsec) / 60 AS connect_duration
                        ,SUM(call_contact_mobile_times) as call_contact_mobile_times
                        ,SUM(call_connect_contact_mobile_times) as call_connect_contact_mobile_times
                        ,SUM(CASE    WHEN is_case_comm = 0 THEN 1 ELSE 0 END) as own_uncomm_case_cnt
                        ,COUNT(DISTINCT 
                              CASE    WHEN ((call_connect_times) > 0) AND (case_status <> 2 OR SUBSTR(trans_time,1,10) <> dt)
                              THEN global_case_id END
                        ) AS call_connect_case_cnt
                        ,COUNT(DISTINCT CASE    WHEN ((call_times) > 0) AND (case_status <> 2 OR SUBSTR(trans_time,1,10) <> dt) 
                        THEN global_case_id END) AS call_case_cnt
                FROM    (select *
                        ,sum(call_times) over (partition by dt) as call_times_check
                        -- ,sum(call_times) over (partition by dt, owner_id) as call_times_agent_check
                        ,MAX(HOUR(last_call_time) + MINUTE(last_call_time) / 60) OVER (PARTITION BY dt,owner_id)-MIN(HOUR(first_call_time) + MINUTE(first_call_time) / 60) OVER (PARTITION BY dt,owner_id) as work_hour
                        from phl_data.dws_fact_coll_case_process_info_daily
                        WHERE   dt >= '2024-10-20'
                        AND     TO_CHAR(DATEADD(TO_DATE(dt),-overdue_days_dt,'day'),'yyyymm') BETWEEN '{start_duemon}' AND '{end_duemon}'
                        AND     owner_bucket IN ('T2','T4')
                        )
                where call_times_check > 2000
                GROUP BY case when case_type = 0 then 'T0'
                            when case_type = 1 AND overdue_days_dt <= 7 then 'T1'
                            when case_type = 1 AND overdue_days_dt between 8 and 15 then 'T2'
                            when case_type = 1 AND overdue_days_dt between 16 and 30 then 'TM1'
                            END
                         ,user_type
                         ,dt
                         ,owner_id
                         ,TO_CHAR(DATEADD(TO_DATE(dt),-overdue_days_dt,'day'),'yyyymm')
            ) a
    GROUP BY owner_bucket
             ,user_type
             ,due_mon
    ;
'''
df = run_sql(sql)
df_duemon_process = generate_bucket(df.sort_values(by=['user_type', 'due_mon'],ascending=[True,False]), ['T0','T1','T2','TM1'])

log_view_address: http://logview.odps.aliyun.com/logview/?h=https://service.ap-southeast-1-vpc.maxcompute.aliyun-inc.com/api&p=phl_anls&i=20260203115921500gah6xc48hrs5&token=NW1ybFR3TmtMdGhXdWMvNjI1T0xIZ05TZTBrPSxPRFBTX09CTzpwNF8yNjE4MzAxNzM0MTg2MTQ1NDcsMTc3MjcxMTk2MSx7IlN0YXRlbWVudCI6W3siQWN0aW9uIjpbIm9kcHM6UmVhZCJdLCJFZmZlY3QiOiJBbGxvdyIsIlJlc291cmNlIjpbImFjczpvZHBzOio6cHJvamVjdHMvcGhsX2FubHMvaW5zdGFuY2VzLzIwMjYwMjAzMTE1OTIxNTAwZ2FoNnhjNDhocnM1Il19XSwiVmVyc2lvbiI6IjEifQ==


In [43]:
df_duemon_process

,owner_bucket,user_type,due_mon,owner_cnt_avg,owing_case_avg,call_connect_times_pp_avg,call_case_pct,call_duration_avg_owner,cover_times,call_times_avg,...,call_duration_avg_owner,cover_times,call_times_avg,call_duration_avg,call_contact_mobile_times_avg,case_connect_times_avg,case_connect_rate,call_connect_rate,contact_call_connect_rate,connect_duration_case_avg
0,T0,新客,202507,38.413793,7.526032,4.485637,0.855916,,6.089351,5.908874,...,,3.458706,1.677239,0.632552,0.377243,0.054093,0.097909,0.032251,0.056893,0.996080
1,T0,新客,202506,33.346154,9.470588,6.277970,0.816222,,5.737261,5.230179,...,,3.578535,1.692107,0.637772,0.415490,0.056965,0.107382,0.033665,0.056165,0.934501
2,T0,新客,202505,36.071429,10.728713,7.567327,0.853728,,5.268053,5.046604,...,,3.107374,1.749527,0.662375,0.530196,0.071206,0.112346,0.040700,0.063497,1.026940
3,T0,新客,202504,36.000000,10.435897,7.863248,0.889844,,5.910464,5.809889,...,,2.474199,1.097016,0.405089,0.312108,0.049366,0.100097,0.045000,0.067901,0.994312
4,T0,新客,202503,29.454545,11.913580,7.254115,0.837565,,4.350447,4.189896,...,,3.701633,2.345677,0.880567,0.648153,0.090436,0.117297,0.038554,0.058377,0.990603
5,T0,新客,202502,32.724138,12.598525,5.814542,0.742556,,3.168596,2.801522,...,,3.248749,1.483439,0.561811,0.474294,0.057545,0.106383,0.038792,0.055464,1.120130
6,T0,新客,202501,20.857143,21.323630,8.116438,0.621778,,2.319926,1.824861,...,,3.640455,2.526617,0.970155,0.849837,0.100601,0.121092,0.039816,0.063915,1.277319
7,T0,老客,202507,129.290323,14.604541,8.150200,0.819868,,16.908870,15.898078,...,,5.001735,2.749144,1.023329,0.662969,0.067617,0.105554,0.024596,0.048525,1.050614
8,T0,老客,202506,125.571429,14.356940,8.645336,0.840686,,14.956810,14.124705,...,,4.932820,2.557170,0.937627,0.669281,0.064381,0.106874,0.025177,0.044454,1.029096
9,T0,老客,202505,123.700000,15.920237,9.959580,0.844770,,11.665030,11.074475,...,,4.231804,2.539297,0.933090,0.805551,0.082708,0.116532,0.032571,0.054707,1.084163


In [44]:
# !!NEW!!
# user_type+flag_principal
sql = f'''
    SELECT  owner_bucket
            ,user_type
            ,flag_owing_principal_alloc
            
            ,sum(owner_cnt) / count(distinct dt) AS owner_cnt_avg
            ,SUM(owing_case_cnt) / SUM(owner_cnt) AS owing_case_avg
            ,sum(call_connect_times) / sum(owner_cnt) AS call_connect_times_pp_avg
            ,sum(call_case_cnt) / sum(owing_case_cnt) as call_case_pct
            ,'' as call_duration_avg_owner
            ,case when sum(owing_case_cnt)>0 then sum(comm_times)/(sum(owing_case_cnt)-sum(own_uncomm_case_cnt)) end as cover_times
            ,SUM(call_times) / SUM(owing_case_cnt) AS call_times_avg
            ,SUM(call_duration) / SUM(owing_case_cnt) AS call_duration_avg
            ,SUM(call_contact_mobile_times) / sum(owing_case_cnt) as call_contact_mobile_times_avg
            
            ,SUM(call_connect_times) / SUM(owing_case_cnt) AS case_connect_times_avg
            ,SUM(call_connect_case_cnt) / SUM(call_case_cnt) AS case_connect_rate
            ,SUM(call_connect_times) / SUM(call_times) AS call_connect_rate
            ,sum(call_connect_contact_mobile_times) / sum(call_contact_mobile_times) as contact_call_connect_rate
            ,SUM(connect_duration) / SUM(call_connect_case_cnt) as connect_duration_case_avg
    FROM    (
                SELECT  case when case_type = 0 then 'T0'
                            when case_type = 1 AND overdue_days_dt <= 7 then 'T1'
                            when case_type = 1 AND overdue_days_dt between 8 and 15 then 'T2'
                            when case_type = 1 AND overdue_days_dt between 16 and 30 then 'TM1'
                            END AS owner_bucket
                        ,user_type
                        ,dt
                        ,case when owing_principal_alloc < 500 then '1.0-500'
                            when owing_principal_alloc < 1000 then '2.500-1000'
                            when owing_principal_alloc < 1500 then '3.1000-1500'
                            when owing_principal_alloc < 2500 then '4.1500-2500'
                            when owing_principal_alloc < 3500 then '5.2500-3500'
                            else '6.3500+' end 
                            as flag_owing_principal_alloc
                        ,count(distinct case when work_hour >= 8 then owner_name end) AS owner_cnt
                        ,COUNT(DISTINCT global_case_id) AS case_cnt
                        ,SUM(CASE    WHEN (case_status <> 2 OR SUBSTR(trans_time,1,10) <> dt) THEN 1 ELSE 0 END) AS owing_case_cnt
                        ,SUM(call_times) AS call_times
                        ,sum(comm_times) as comm_times
                        ,SUM(call_duration) / 60 AS call_duration
                        ,SUM(call_connect_times) AS call_connect_times
                        ,SUM(call_billsec) / 60 AS connect_duration
                        ,SUM(call_contact_mobile_times) as call_contact_mobile_times
                        ,SUM(call_connect_contact_mobile_times) as call_connect_contact_mobile_times
                        ,SUM(CASE    WHEN is_case_comm = 0 THEN 1 ELSE 0 END) as own_uncomm_case_cnt
                        ,COUNT(DISTINCT 
                              CASE    WHEN ((call_connect_times) > 0) AND (case_status <> 2 OR SUBSTR(trans_time,1,10) <> dt)
                              THEN global_case_id END
                        ) AS call_connect_case_cnt
                        ,COUNT(DISTINCT CASE    WHEN ((call_times) > 0) AND (case_status <> 2 OR SUBSTR(trans_time,1,10) <> dt) 
                        THEN global_case_id END) AS call_case_cnt
                FROM    (select *
                        ,sum(call_times) over (partition by dt) as call_times_check
                        -- ,sum(call_times) over (partition by dt, owner_id) as call_times_agent_check
                        ,MAX(HOUR(last_call_time) + MINUTE(last_call_time) / 60) OVER (PARTITION BY dt,owner_id)-MIN(HOUR(first_call_time) + MINUTE(first_call_time) / 60) OVER (PARTITION BY dt,owner_id) as work_hour
                        from phl_data.dws_fact_coll_case_process_info_daily
                        WHERE   dt >= '2024-10-20'
                        AND     TO_CHAR(DATEADD(TO_DATE(dt),-overdue_days_dt,'day'),'yyyymm') BETWEEN '{start_duemon}' AND '{end_duemon}'
                        AND     owner_bucket IN ('T2','T4')
                        )
                where call_times_check > 2000
                GROUP BY case when case_type = 0 then 'T0'
                            when case_type = 1 AND overdue_days_dt <= 7 then 'T1'
                            when case_type = 1 AND overdue_days_dt between 8 and 15 then 'T2'
                            when case_type = 1 AND overdue_days_dt between 16 and 30 then 'TM1'
                            END
                         ,user_type
                         ,dt
                         ,owner_id
                        ,case when owing_principal_alloc < 500 then '1.0-500'
                            when owing_principal_alloc < 1000 then '2.500-1000'
                            when owing_principal_alloc < 1500 then '3.1000-1500'
                            when owing_principal_alloc < 2500 then '4.1500-2500'
                            when owing_principal_alloc < 3500 then '5.2500-3500'
                            else '6.3500+' end 
            ) a
    GROUP BY owner_bucket
             ,user_type
             ,flag_owing_principal_alloc
    ;
'''
df = run_sql(sql)
df_principal_process = generate_bucket(df.sort_values(by=['user_type', 'flag_owing_principal_alloc']), ['T0','T1','T2','TM1'])

log_view_address: http://logview.odps.aliyun.com/logview/?h=https://service.ap-southeast-1-vpc.maxcompute.aliyun-inc.com/api&p=phl_anls&i=20260203115937637gak6xc48hrs5&token=QWFTQ2lwZXJQckV0MnBab3Z6K0x1KzNoNVFJPSxPRFBTX09CTzpwNF8yNjE4MzAxNzM0MTg2MTQ1NDcsMTc3MjcxMTk3Nyx7IlN0YXRlbWVudCI6W3siQWN0aW9uIjpbIm9kcHM6UmVhZCJdLCJFZmZlY3QiOiJBbGxvdyIsIlJlc291cmNlIjpbImFjczpvZHBzOio6cHJvamVjdHMvcGhsX2FubHMvaW5zdGFuY2VzLzIwMjYwMjAzMTE1OTM3NjM3Z2FrNnhjNDhocnM1Il19XSwiVmVyc2lvbiI6IjEifQ==


In [45]:
# !!NEW!!
# user_type+flag_principal
sql = f'''
    SELECT  owner_bucket
            ,user_type
            ,flag_owing_principal_alloc
            
            ,sum(owner_cnt) / count(distinct dt) AS owner_cnt_avg
            ,SUM(owing_case_cnt) / SUM(owner_cnt) AS owing_case_avg
            ,sum(call_connect_times) / sum(owner_cnt) AS call_connect_times_pp_avg
            ,sum(call_case_cnt) / sum(owing_case_cnt) as call_case_pct
            ,'' as call_duration_avg_owner
            ,case when sum(owing_case_cnt)>0 then sum(comm_times)/(sum(owing_case_cnt)-sum(own_uncomm_case_cnt)) end as cover_times
            ,SUM(call_times) / SUM(owing_case_cnt) AS call_times_avg
            ,SUM(call_duration) / SUM(owing_case_cnt) AS call_duration_avg
            ,SUM(call_contact_mobile_times) / sum(owing_case_cnt) as call_contact_mobile_times_avg
            
            ,SUM(call_connect_times) / SUM(owing_case_cnt) AS case_connect_times_avg
            ,SUM(call_connect_case_cnt) / SUM(call_case_cnt) AS case_connect_rate
            ,SUM(call_connect_times) / SUM(call_times) AS call_connect_rate
            ,sum(call_connect_contact_mobile_times) / sum(call_contact_mobile_times) as contact_call_connect_rate
            ,SUM(connect_duration) / SUM(call_connect_case_cnt) as connect_duration_case_avg
    FROM    (
                SELECT  case when case_type = 0 then 'T0'
                            when case_type = 1 AND overdue_days_dt <= 7 then 'T1'
                            when case_type = 1 AND overdue_days_dt between 8 and 15 then 'T2'
                            when case_type = 1 AND overdue_days_dt between 16 and 30 then 'TM1'
                            END AS owner_bucket
                        ,user_type
                        ,dt
                        ,case when owing_principal_alloc < 500 then '1.0-500'
                            when owing_principal_alloc < 1000 then '2.500-1000'
                            when owing_principal_alloc < 1500 then '3.1000-1500'
                            when owing_principal_alloc < 2500 then '4.1500-2500'
                            when owing_principal_alloc < 3500 then '5.2500-3500'
                            else '6.3500+' end 
                            as flag_owing_principal_alloc
                        ,count(distinct case when work_hour >= 8 then owner_name end) AS owner_cnt
                        ,COUNT(DISTINCT global_case_id) AS case_cnt
                        ,SUM(CASE    WHEN (case_status <> 2 OR SUBSTR(trans_time,1,10) <> dt) THEN 1 ELSE 0 END) AS owing_case_cnt
                        ,SUM(call_times) AS call_times
                        ,sum(comm_times) as comm_times
                        ,SUM(call_duration) / 60 AS call_duration
                        ,SUM(call_connect_times) AS call_connect_times
                        ,SUM(call_billsec) / 60 AS connect_duration
                        ,SUM(call_contact_mobile_times) as call_contact_mobile_times
                        ,SUM(call_connect_contact_mobile_times) as call_connect_contact_mobile_times
                        ,SUM(CASE    WHEN is_case_comm = 0 THEN 1 ELSE 0 END) as own_uncomm_case_cnt
                        ,COUNT(DISTINCT 
                              CASE    WHEN ((call_connect_times) > 0) AND (case_status <> 2 OR SUBSTR(trans_time,1,10) <> dt)
                              THEN global_case_id END
                        ) AS call_connect_case_cnt
                        ,COUNT(DISTINCT CASE    WHEN ((call_times) > 0) AND (case_status <> 2 OR SUBSTR(trans_time,1,10) <> dt) 
                        THEN global_case_id END) AS call_case_cnt
                FROM    (select *
                        ,sum(call_times) over (partition by dt) as call_times_check
                        -- ,sum(call_times) over (partition by dt, owner_id) as call_times_agent_check
                        ,MAX(HOUR(last_call_time) + MINUTE(last_call_time) / 60) OVER (PARTITION BY dt,owner_id)-MIN(HOUR(first_call_time) + MINUTE(first_call_time) / 60) OVER (PARTITION BY dt,owner_id) as work_hour
                        from phl_data.dws_fact_coll_case_process_info_daily
                        WHERE   dt >= '2024-10-20'
                        AND     TO_CHAR(DATEADD(TO_DATE(dt),-overdue_days_dt,'day'),'yyyymm') BETWEEN '{start_duemon}' AND '{end_duemon}'
                        AND     owner_bucket IN ('T2','T4')
                        )
                where call_times_check > 2000
                GROUP BY case when case_type = 0 then 'T0'
                            when case_type = 1 AND overdue_days_dt <= 7 then 'T1'
                            when case_type = 1 AND overdue_days_dt between 8 and 15 then 'T2'
                            when case_type = 1 AND overdue_days_dt between 16 and 30 then 'TM1'
                            END
                         ,user_type
                         ,dt
                         ,owner_id
                        ,case when owing_principal_alloc < 500 then '1.0-500'
                            when owing_principal_alloc < 1000 then '2.500-1000'
                            when owing_principal_alloc < 1500 then '3.1000-1500'
                            when owing_principal_alloc < 2500 then '4.1500-2500'
                            when owing_principal_alloc < 3500 then '5.2500-3500'
                            else '6.3500+' end 
            ) a
    GROUP BY owner_bucket
             ,user_type
             ,flag_owing_principal_alloc
    ;
'''
df = run_sql(sql)
df_principal_process = generate_bucket(df.sort_values(by=['user_type', 'flag_owing_principal_alloc']), ['T0','T1','T2','TM1'])

log_view_address: http://logview.odps.aliyun.com/logview/?h=https://service.ap-southeast-1-vpc.maxcompute.aliyun-inc.com/api&p=phl_anls&i=20260203115958616gu1t0l1x9l7&token=OVVrUko5RklIT2t1YnFuRWIwbkhxU0QwYU1nPSxPRFBTX09CTzpwNF8yNjE4MzAxNzM0MTg2MTQ1NDcsMTc3MjcxMTk5OCx7IlN0YXRlbWVudCI6W3siQWN0aW9uIjpbIm9kcHM6UmVhZCJdLCJFZmZlY3QiOiJBbGxvdyIsIlJlc291cmNlIjpbImFjczpvZHBzOio6cHJvamVjdHMvcGhsX2FubHMvaW5zdGFuY2VzLzIwMjYwMjAzMTE1OTU4NjE2Z3UxdDBsMXg5bDciXX1dLCJWZXJzaW9uIjoiMSJ9


### 2.4 TT mob×过程指标

In [46]:
# mob过程指标
sql=f'''
WITH tmp AS 
(
    SELECT  mob, modelbin_a, modelbin_c
        ,a.*
    FROM   (
                    SELECT  DATEADD(TO_DATE(dt),-overdue_days_dt,'day') AS due_date
                            ,case when case_type = 0 then 'T0'
                                when case_type = 1 AND overdue_days_dt <= 7 then 'T1'
                                when case_type = 1 AND overdue_days_dt between 8 and 15 then 'T2'
                                when case_type = 1 AND overdue_days_dt between 16 and 30 then 'TM1'
                                END AS owner_bucket
                            ,user_type
                            ,owner_id
                            ,borrower_id
                            ,global_case_id
                            ,dt
                            ,call_contact_mobile_times 
                            ,call_connect_contact_mobile_times
                            ,call_contact_mobile_billsec
                            ,comm_times
                            ,call_times
                            ,call_connect_times
                            ,call_duration
                            ,call_ring_duration
                            ,call_billsec
                            ,call_mobile_cnt
                            ,call_connect_mobile_cnt
                            ,is_case_comm
                            ,case_status
                            ,trans_time
                            ,sum(call_times) over (partition by dt) as call_times_check
                            -- ,sum(call_times) over (partition by dt, owner_id) as call_times_agent_check
                            ,MAX(HOUR(last_call_time) + MINUTE(last_call_time) / 60) OVER (PARTITION BY dt,owner_id)-MIN(HOUR(first_call_time) + MINUTE(first_call_time) / 60) OVER (PARTITION BY dt,owner_id) as work_hour
                    FROM    phl_data.dws_fact_coll_case_process_info_daily
                    WHERE   dt >= '2024-10-20'
                    AND     TO_CHAR(DATEADD(TO_DATE(dt),-overdue_days_dt,'day'),'yyyymm') BETWEEN '{start_duemon}' AND '{end_duemon}'
                    AND     owner_bucket IN ('T2','T4')
                ) a
    LEFT JOIN    (
            SELECT  user_id
                    ,due_date
                    ,mob_tt as mob
                    ,case when modelbinnew >='E' then 'E+' else modelbinnew end as modelbin_a
                    ,case when s0_bucket in ('B01','B02') then 'E'
                        when s0_bucket in ('B03','B04','B05') then 'D'
                        when s0_bucket in ('B06','B07','B08','B09') then 'C'
                        when s0_bucket in ('B10','B11','B12','B13') then 'B'
                        else 'A'
                        end as modelbin_c
            FROM    phl_anls.tmp_zyt_tt_vintage
            WHERE   TO_CHAR(due_date,'yyyymm') BETWEEN '{start_duemon}' AND '{end_duemon}'
            and model_bin is not null
        ) v
    ON      v.user_id = a.borrower_id
    AND     v.due_date = a.due_date
)


SELECT  mob
        ,owner_bucket

            
        ,sum(owner_cnt) / count(distinct dt) AS owner_cnt_avg
        ,SUM(owing_case_cnt) / SUM(owner_cnt) AS owing_case_avg
        ,sum(call_connect_times) / sum(owner_cnt) AS call_connect_times_pp_avg
        ,sum(call_case_cnt) / sum(owing_case_cnt) as call_case_pct
        ,'' as call_duration_avg_owner
        ,case when sum(owing_case_cnt)>0 then sum(comm_times)/(sum(owing_case_cnt)-sum(own_uncomm_case_cnt)) end as cover_times
        ,SUM(call_times) / SUM(owing_case_cnt) AS call_times_avg
        ,SUM(call_duration) / SUM(owing_case_cnt) AS call_duration_avg
        ,SUM(call_contact_mobile_times) / sum(owing_case_cnt) as call_contact_mobile_times_avg

        ,SUM(call_connect_times) / SUM(owing_case_cnt) AS case_connect_times_avg
        ,SUM(call_connect_case_cnt) / SUM(call_case_cnt) AS case_connect_rate
        ,SUM(call_connect_times) / SUM(call_times) AS call_connect_rate
        ,sum(call_connect_contact_mobile_times) / sum(call_contact_mobile_times) as contact_call_connect_rate
        ,SUM(connect_duration) / SUM(call_connect_case_cnt) as connect_duration_case_avg
FROM    (
            SELECT  owner_bucket
                    ,mob

                    ,dt
                    ,count(distinct case when work_hour >= 8 then owner_id end) AS owner_cnt
                    ,COUNT(DISTINCT global_case_id) AS case_cnt
                    ,SUM(CASE    WHEN (case_status <> 2 OR SUBSTR(trans_time,1,10) <> dt) THEN 1 ELSE 0 END) AS owing_case_cnt
                    ,SUM(call_times) AS call_times
                    ,sum(comm_times) as comm_times
                    ,SUM(call_duration) / 60 AS call_duration
                    ,SUM(call_connect_times) AS call_connect_times
                    ,SUM(call_billsec) / 60 AS connect_duration
                    ,SUM(call_contact_mobile_times) as call_contact_mobile_times
                    ,SUM(call_connect_contact_mobile_times) as call_connect_contact_mobile_times
                    ,SUM(CASE    WHEN is_case_comm = 0 THEN 1 ELSE 0 END) as own_uncomm_case_cnt
                    ,COUNT(DISTINCT 
                          CASE    WHEN ((call_connect_times) > 0) AND (case_status <> 2 OR SUBSTR(trans_time,1,10) <> dt)
                          THEN global_case_id END
                    ) AS call_connect_case_cnt
                    ,COUNT(DISTINCT CASE    WHEN ((call_times) > 0) AND (case_status <> 2 OR SUBSTR(trans_time,1,10) <> dt) 
                    THEN global_case_id END) AS call_case_cnt
            FROM    tmp
            WHERE   call_times_check > 2000
            GROUP BY owner_bucket, mob
                     ,dt
                     ,owner_id
        ) 
GROUP BY owner_bucket, mob
;
    '''
df = run_sql(sql)
df_mob_process = df[df['mob'].notna()].sort_values('mob')
df_mob_process = generate_bucket(df_mob_process, ['T0','T1','T2','TM1'])

log_view_address: http://logview.odps.aliyun.com/logview/?h=https://service.ap-southeast-1-vpc.maxcompute.aliyun-inc.com/api&p=phl_anls&i=20260203120021515gzlykot1al7&token=enVKdldkc25wWlI3WlNXUTd1RTV1SzdXcEhnPSxPRFBTX09CTzpwNF8yNjE4MzAxNzM0MTg2MTQ1NDcsMTc3MjcxMjAyMSx7IlN0YXRlbWVudCI6W3siQWN0aW9uIjpbIm9kcHM6UmVhZCJdLCJFZmZlY3QiOiJBbGxvdyIsIlJlc291cmNlIjpbImFjczpvZHBzOio6cHJvamVjdHMvcGhsX2FubHMvaW5zdGFuY2VzLzIwMjYwMjAzMTIwMDIxNTE1Z3pseWtvdDFhbDciXX1dLCJWZXJzaW9uIjoiMSJ9


In [47]:
# modelbin a过程指标
sql=f'''
WITH tmp AS 
(
    SELECT  mob, modelbin_a, modelbin_c
        ,a.*
    FROM   (
                    SELECT  DATEADD(TO_DATE(dt),-overdue_days_dt,'day') AS due_date
                            ,case when case_type = 0 then 'T0'
                                when case_type = 1 AND overdue_days_dt <= 7 then 'T1'
                                when case_type = 1 AND overdue_days_dt between 8 and 15 then 'T2'
                                when case_type = 1 AND overdue_days_dt between 16 and 30 then 'TM1'
                                END AS owner_bucket
                            ,user_type
                            ,owner_id
                            ,borrower_id
                            ,global_case_id
                            ,dt
                            ,call_contact_mobile_times 
                            ,call_connect_contact_mobile_times
                            ,call_contact_mobile_billsec
                            ,comm_times
                            ,call_times
                            ,call_connect_times
                            ,call_duration
                            ,call_ring_duration
                            ,call_billsec
                            ,call_mobile_cnt
                            ,call_connect_mobile_cnt
                            ,is_case_comm
                            ,case_status
                            ,trans_time
                            ,sum(call_times) over (partition by dt) as call_times_check
                            -- ,sum(call_times) over (partition by dt, owner_id) as call_times_agent_check
                            ,MAX(HOUR(last_call_time) + MINUTE(last_call_time) / 60) OVER (PARTITION BY dt,owner_id)-MIN(HOUR(first_call_time) + MINUTE(first_call_time) / 60) OVER (PARTITION BY dt,owner_id) as work_hour
                    FROM    phl_data.dws_fact_coll_case_process_info_daily
                    WHERE   dt >= '2024-10-20'
                    AND     TO_CHAR(DATEADD(TO_DATE(dt),-overdue_days_dt,'day'),'yyyymm') BETWEEN '{start_duemon}' AND '{end_duemon}'
                    AND     owner_bucket IN ('T2','T4')
                ) a
    LEFT JOIN    (
            SELECT  user_id
                    ,due_date
                    ,mob_tt as mob
                    ,case when modelbinnew >='E' then 'E+' else modelbinnew end as modelbin_a
                    ,case when s0_bucket in ('B01','B02') then 'E'
                        when s0_bucket in ('B03','B04','B05') then 'D'
                        when s0_bucket in ('B06','B07','B08','B09') then 'C'
                        when s0_bucket in ('B10','B11','B12','B13') then 'B'
                        else 'A'
                        end as modelbin_c
            FROM    phl_anls.tmp_zyt_tt_vintage
            WHERE   TO_CHAR(due_date,'yyyymm') BETWEEN '{start_duemon}' AND '{end_duemon}'
        ) v
    ON      v.user_id = a.borrower_id
    AND     v.due_date = a.due_date
)


SELECT modelbin_a
        ,owner_bucket
        ,user_type
        ,sum(owner_cnt) / count(distinct dt) AS owner_cnt_avg
        ,SUM(owing_case_cnt) / SUM(owner_cnt) AS owing_case_avg
        ,sum(call_connect_times) / sum(owner_cnt) AS call_connect_times_pp_avg
        ,sum(call_case_cnt) / sum(owing_case_cnt) as call_case_pct
        ,'' as call_duration_avg_owner
        ,case when sum(owing_case_cnt)>0 then sum(comm_times)/(sum(owing_case_cnt)-sum(own_uncomm_case_cnt)) end as cover_times
        ,SUM(call_times) / SUM(owing_case_cnt) AS call_times_avg
        ,SUM(call_duration) / SUM(owing_case_cnt) AS call_duration_avg
        ,SUM(call_contact_mobile_times) / sum(owing_case_cnt) as call_contact_mobile_times_avg

        ,SUM(call_connect_times) / SUM(owing_case_cnt) AS case_connect_times_avg
        ,SUM(call_connect_case_cnt) / SUM(call_case_cnt) AS case_connect_rate
        ,SUM(call_connect_times) / SUM(call_times) AS call_connect_rate
        ,sum(call_connect_contact_mobile_times) / sum(call_contact_mobile_times) as contact_call_connect_rate
        ,SUM(connect_duration) / SUM(call_connect_case_cnt) as connect_duration_case_avg
FROM    (
            SELECT  owner_bucket
                    ,modelbin_a
                    ,user_type
                    ,dt
                    ,count(distinct case when work_hour >= 8 then owner_id end) AS owner_cnt
                    ,COUNT(DISTINCT global_case_id) AS case_cnt
                    ,SUM(CASE    WHEN (case_status <> 2 OR SUBSTR(trans_time,1,10) <> dt) THEN 1 ELSE 0 END) AS owing_case_cnt
                    ,SUM(call_times) AS call_times
                    ,sum(comm_times) as comm_times
                    ,SUM(call_duration) / 60 AS call_duration
                    ,SUM(call_connect_times) AS call_connect_times
                    ,SUM(call_billsec) / 60 AS connect_duration
                    ,SUM(call_contact_mobile_times) as call_contact_mobile_times
                    ,SUM(call_connect_contact_mobile_times) as call_connect_contact_mobile_times
                    ,SUM(CASE    WHEN is_case_comm = 0 THEN 1 ELSE 0 END) as own_uncomm_case_cnt
                    ,COUNT(DISTINCT 
                          CASE    WHEN ((call_connect_times) > 0) AND (case_status <> 2 OR SUBSTR(trans_time,1,10) <> dt)
                          THEN global_case_id END
                    ) AS call_connect_case_cnt
                    ,COUNT(DISTINCT CASE    WHEN ((call_times) > 0) AND (case_status <> 2 OR SUBSTR(trans_time,1,10) <> dt) 
                    THEN global_case_id END) AS call_case_cnt
            FROM    tmp
            WHERE   call_times_check > 2000
            GROUP BY owner_bucket, modelbin_a, user_type
                     ,dt
                     ,owner_id
        ) 
GROUP BY owner_bucket, user_type, modelbin_a
;
    '''
df = run_sql(sql)
df_modelbina_process = df[df['modelbin_a'].notna()].sort_values(['user_type','modelbin_a'])
df_modelbina_process = generate_bucket(df_modelbina_process, ['T0','T1','T2','TM1'])

log_view_address: http://logview.odps.aliyun.com/logview/?h=https://service.ap-southeast-1-vpc.maxcompute.aliyun-inc.com/api&p=phl_anls&i=20260203120122834gwruy851zw6&token=UHBZQmxLZWl1WTNaQVVYbS9QWUNiRlA3aC84PSxPRFBTX09CTzpwNF8yNjE4MzAxNzM0MTg2MTQ1NDcsMTc3MjcxMjA4Mix7IlN0YXRlbWVudCI6W3siQWN0aW9uIjpbIm9kcHM6UmVhZCJdLCJFZmZlY3QiOiJBbGxvdyIsIlJlc291cmNlIjpbImFjczpvZHBzOio6cHJvamVjdHMvcGhsX2FubHMvaW5zdGFuY2VzLzIwMjYwMjAzMTIwMTIyODM0Z3dydXk4NTF6dzYiXX1dLCJWZXJzaW9uIjoiMSJ9


In [48]:
# modelbin c过程指标
sql=f'''
WITH tmp AS 
(
    SELECT  mob, modelbin_a, modelbin_c
        ,a.*
    FROM   (
                    SELECT  DATEADD(TO_DATE(dt),-overdue_days_dt,'day') AS due_date
                            ,case when case_type = 0 then 'T0'
                                when case_type = 1 AND overdue_days_dt <= 7 then 'T1'
                                when case_type = 1 AND overdue_days_dt between 8 and 15 then 'T2'
                                when case_type = 1 AND overdue_days_dt between 16 and 30 then 'TM1'
                                END AS owner_bucket
                            ,user_type
                            ,owner_id
                            ,borrower_id
                            ,global_case_id
                            ,dt
                            ,call_contact_mobile_times 
                            ,call_connect_contact_mobile_times
                            ,call_contact_mobile_billsec
                            ,comm_times
                            ,call_times
                            ,call_connect_times
                            ,call_duration
                            ,call_ring_duration
                            ,call_billsec
                            ,call_mobile_cnt
                            ,call_connect_mobile_cnt
                            ,is_case_comm
                            ,case_status
                            ,trans_time
                            ,sum(call_times) over (partition by dt) as call_times_check
                            -- ,sum(call_times) over (partition by dt, owner_id) as call_times_agent_check
                            ,MAX(HOUR(last_call_time) + MINUTE(last_call_time) / 60) OVER (PARTITION BY dt,owner_id)-MIN(HOUR(first_call_time) + MINUTE(first_call_time) / 60) OVER (PARTITION BY dt,owner_id) as work_hour
                    FROM    phl_data.dws_fact_coll_case_process_info_daily
                    WHERE   dt >= '2024-10-20'
                    AND     TO_CHAR(DATEADD(TO_DATE(dt),-overdue_days_dt,'day'),'yyyymm') BETWEEN '{start_duemon}' AND '{end_duemon}'
                    AND     owner_bucket IN ('T2','T4')
                ) a
    LEFT JOIN    (
            SELECT  user_id
                    ,due_date
                    ,mob_tt as mob
                    ,case when modelbinnew >='E' then 'E+' else modelbinnew end as modelbin_a
                    ,case when s0_bucket in ('B01','B02') then 'E'
                        when s0_bucket in ('B03','B04','B05') then 'D'
                        when s0_bucket in ('B06','B07','B08','B09') then 'C'
                        when s0_bucket in ('B10','B11','B12','B13') then 'B'
                        else 'A'
                        end as modelbin_c
            FROM    phl_anls.tmp_zyt_tt_vintage
            WHERE   TO_CHAR(due_date,'yyyymm') BETWEEN '{start_duemon}' AND '{end_duemon}'
        ) v
    ON      v.user_id = a.borrower_id
    AND     v.due_date = a.due_date
)


SELECT modelbin_c
        ,owner_bucket
        ,user_type
        ,sum(owner_cnt) / count(distinct dt) AS owner_cnt_avg
        ,SUM(owing_case_cnt) / SUM(owner_cnt) AS owing_case_avg
        ,sum(call_connect_times) / sum(owner_cnt) AS call_connect_times_pp_avg
        ,sum(call_case_cnt) / sum(owing_case_cnt) as call_case_pct
        ,'' as call_duration_avg_owner
        ,case when sum(owing_case_cnt)>0 then sum(comm_times)/(sum(owing_case_cnt)-sum(own_uncomm_case_cnt)) end as cover_times
        ,SUM(call_times) / SUM(owing_case_cnt) AS call_times_avg
        ,SUM(call_duration) / SUM(owing_case_cnt) AS call_duration_avg
        ,SUM(call_contact_mobile_times) / sum(owing_case_cnt) as call_contact_mobile_times_avg

        ,SUM(call_connect_times) / SUM(owing_case_cnt) AS case_connect_times_avg
        ,SUM(call_connect_case_cnt) / SUM(call_case_cnt) AS case_connect_rate
        ,SUM(call_connect_times) / SUM(call_times) AS call_connect_rate
        ,sum(call_connect_contact_mobile_times) / sum(call_contact_mobile_times) as contact_call_connect_rate
        ,SUM(connect_duration) / SUM(call_connect_case_cnt) as connect_duration_case_avg
FROM    (
            SELECT  owner_bucket
                    ,modelbin_c
                    ,user_type
                    ,dt
                    ,count(distinct case when work_hour >= 8 then owner_id end) AS owner_cnt
                    ,COUNT(DISTINCT global_case_id) AS case_cnt
                    ,SUM(CASE    WHEN (case_status <> 2 OR SUBSTR(trans_time,1,10) <> dt) THEN 1 ELSE 0 END) AS owing_case_cnt
                    ,SUM(call_times) AS call_times
                    ,sum(comm_times) as comm_times
                    ,SUM(call_duration) / 60 AS call_duration
                    ,SUM(call_connect_times) AS call_connect_times
                    ,SUM(call_billsec) / 60 AS connect_duration
                    ,SUM(call_contact_mobile_times) as call_contact_mobile_times
                    ,SUM(call_connect_contact_mobile_times) as call_connect_contact_mobile_times
                    ,SUM(CASE    WHEN is_case_comm = 0 THEN 1 ELSE 0 END) as own_uncomm_case_cnt
                    ,COUNT(DISTINCT 
                          CASE    WHEN ((call_connect_times) > 0) AND (case_status <> 2 OR SUBSTR(trans_time,1,10) <> dt)
                          THEN global_case_id END
                    ) AS call_connect_case_cnt
                    ,COUNT(DISTINCT CASE    WHEN ((call_times) > 0) AND (case_status <> 2 OR SUBSTR(trans_time,1,10) <> dt) 
                    THEN global_case_id END) AS call_case_cnt
            FROM    tmp
            WHERE   call_times_check > 2000
            GROUP BY owner_bucket, modelbin_c, user_type
                     ,dt
                     ,owner_id
        ) 
GROUP BY owner_bucket, user_type, modelbin_c
;
    '''
df = run_sql(sql)
df_modelbinc_process = df[(df['modelbin_c'].notna()) & (df['modelbin_c']!='err')].sort_values(['user_type','modelbin_c'])
df_modelbinc_process = generate_bucket(df_modelbinc_process, ['T0','T1','T2','TM1'])

log_view_address: http://logview.odps.aliyun.com/logview/?h=https://service.ap-southeast-1-vpc.maxcompute.aliyun-inc.com/api&p=phl_anls&i=20260203120222176gkmv0l1x9l7&token=bHg1eWlYbGtVWmhlYmlOZDk5aVVVUEMyZ3BNPSxPRFBTX09CTzpwNF8yNjE4MzAxNzM0MTg2MTQ1NDcsMTc3MjcxMjE0Mix7IlN0YXRlbWVudCI6W3siQWN0aW9uIjpbIm9kcHM6UmVhZCJdLCJFZmZlY3QiOiJBbGxvdyIsIlJlc291cmNlIjpbImFjczpvZHBzOio6cHJvamVjdHMvcGhsX2FubHMvaW5zdGFuY2VzLzIwMjYwMjAzMTIwMjIyMTc2Z2ttdjBsMXg5bDciXX1dLCJWZXJzaW9uIjoiMSJ9


In [49]:
with pd.ExcelWriter('output.xlsx',mode='a') as writer:
    df_duemon.to_excel(writer, sheet_name='Tiktok', startrow=2, startcol=0, index=True)
    df_duemon_process.to_excel(writer, sheet_name='Tiktok', startrow=2, startcol=30, index=True)
    
    df_principal.to_excel(writer, sheet_name='Tiktok', startrow=6+df_duemon.shape[0], startcol=0, index=True)
    df_principal_process.to_excel(writer, sheet_name='Tiktok', startrow=6+df_duemon.shape[0], startcol=30, index=True)
    df_principal_duemon_result.to_excel(writer, sheet_name='Tiktok', startrow=10+df_duemon.shape[0]*2, startcol=0, index=True)
    
    df_mob.to_excel(writer, sheet_name='Tiktok', startrow=14+df_duemon.shape[0]*3, startcol=1, index=True)
    df_mob_process.to_excel(writer, sheet_name='Tiktok', startrow=14+df_duemon.shape[0]*3, startcol=30, index=True)
    
    df_modelbina.to_excel(writer, sheet_name='Tiktok', startrow=18+df_duemon.shape[0]*4, startcol=1, index=True)
    df_modelbina_process.to_excel(writer, sheet_name='Tiktok', startrow=18+df_duemon.shape[0]*4, startcol=30, index=True)
    df_modelbina_duemon_result.to_excel(writer, sheet_name='Tiktok', startrow=22+df_duemon.shape[0]*5, startcol=0, index=True)
    
    df_modelbinc.to_excel(writer, sheet_name='Tiktok', startrow=26+df_duemon.shape[0]*6, startcol=1, index=True)
    df_modelbinc_process.to_excel(writer, sheet_name='Tiktok', startrow=26+df_duemon.shape[0]*6, startcol=30, index=True)
    df_modelbinc_duemon_result.to_excel(writer, sheet_name='Tiktok', startrow=30+df_duemon.shape[0]*7, startcol=0, index=True)

ValueError: Append mode is not supported with xlsxwriter!

In [ ]:
!pip install xlsxwriter

In [ ]:
df_total_tt

# Lazada

In [ ]:
sql =f'''
SELECT  case when user_type = '新客' then '新客'
            when user_type in ('新转化老客','存量老客') then '老客'
            else 'total' end as user_type
        ,TO_CHAR(due_date,'yyyymm') AS due_mon
        ,flag_tt_principal
        ,case when modelbinnew >='E' then 'E+' else modelbinnew end as modelbin_a
        ,mob
        ,SUM(owing_user_cnt) as owing_user_cnt
        ,SUM(owing_principal) as owing_principal
        ,'' as cust_pct
        ,sum(owing_principal)/sum(owing_user_cnt) as owing_principal_avg
        ,AVG(period_seq_lzd) as period_seq_avg
        ,SUM(case when datediff(now(),to_date(due_date))>=1 then overdue_principal end)/ SUM(case when datediff(now(),to_date(due_date))>=1 then owing_principal end) AS overdue_rate
        ,case when SUM(case when datediff(now(),to_date(due_date))>=2 then overdue_principal end) = 0 then '/' else 1 - SUM(case when datediff(now(),to_date(due_date))>=2 then d2_principal end)/ SUM(case when datediff(now(),to_date(due_date))>=2 then overdue_principal end) end AS dpd1_repay
        ,case when SUM(case when datediff(now(),to_date(due_date))>=6 then overdue_principal end) = 0 then '/' else 1 - SUM(case when datediff(now(),to_date(due_date))>=6 then d6_principal end)/ SUM(case when datediff(now(),to_date(due_date))>=6 then overdue_principal end) end AS dpd5_repay
        ,case when SUM(case when datediff(now(),to_date(due_date))>=8 then overdue_principal end) = 0 then '/' else 1 - SUM(case when datediff(now(),to_date(due_date))>=8 then d8_principal end)/ SUM(case when datediff(now(),to_date(due_date))>=8 then overdue_principal end) end AS dpd7_repay
        ,case when SUM(case when datediff(now(),to_date(due_date))>=16 then overdue_principal end) = 0 then '/' else 1 - SUM(case when datediff(now(),to_date(due_date))>=16 then d16_principal end)/ SUM(case when datediff(now(),to_date(due_date))>=16 then overdue_principal end) end AS dpd15_repay
        ,case when SUM(case when datediff(now(),to_date(due_date))>=31 then overdue_principal end) = 0 then '/' else 1 - SUM(case when datediff(now(),to_date(due_date))>=31 then d31_principal end)/ SUM(case when datediff(now(),to_date(due_date))>=31 then overdue_principal end) end AS dpd30_repay
        ,SUM(case when datediff(now(),to_date(due_date))>=6 then d6_principal end)/ SUM(case when datediff(now(),to_date(due_date))>=6 then owing_principal end) AS dpd5
        ,case when SUM(case when datediff(now(),to_date(due_date))>=31 then owing_principal end) = 0 then '/' else SUM(case when datediff(now(),to_date(due_date))>=31 then d31_principal end)/ SUM(case when datediff(now(),to_date(due_date))>=31 then owing_principal end) end AS dpd30
        ,case when sum(owing_user_cnt) = 0 then '/' else sum(case when is_touch0=2 then owing_user_cnt end)/sum(owing_user_cnt) end as connect_rate0
        ,case when sum(owing_user_cnt) = 0 then '/' else sum(case when action_code0 like '%PTP' then owing_user_cnt end)/sum(owing_user_cnt) end as ptp_rate0
        ,case when sum(case when is_touch0=2 then overdue_principal end)=0 then '/' else sum(case when is_touch0=2 then overdue_principal-d11_principal end)/sum(case when is_touch0=2 then overdue_principal end) end as connect_conversion0
        ,case when sum(case when action_code0 like '%PTP' then overdue_principal end)=0 then '/' else sum(case when action_code0 like '%PTP' then overdue_principal-d11_principal end)/sum(case when action_code0 like '%PTP' then overdue_principal end) end as ptp_conversion0
        ,case when sum(case when is_touch0=1 then overdue_principal end)=0 then '/' else sum(case when is_touch0=1 then overdue_principal-d11_principal end)/sum(case when is_touch0=1 then overdue_principal end) end as disconnect_conversion0

        ,case when sum(case when is_touch is not null then owing_user_cnt end)=0 then '/' else sum(case when is_touch=2 then owing_user_cnt end)/sum(case when is_touch is not null then owing_user_cnt end) end as connect_rate
        ,case when sum(case when is_touch is not null then owing_user_cnt end)=0 then '/' else sum(case when action_code like '%PTP' then owing_user_cnt end)/sum(case when is_touch is not null then owing_user_cnt end) end as ptp_rate
        ,case when sum(case when is_touch=2 then overdue_principal end)=0 then '/' else sum(case when is_touch=2 then overdue_principal-d11_principal end)/sum(case when is_touch=2 then overdue_principal end) end as connect_conversion
        ,case when sum(case when action_code like '%PTP' then overdue_principal end)=0 then '/' else sum(case when action_code like '%PTP' then overdue_principal-d11_principal end)/sum(case when action_code like '%PTP' then overdue_principal end) end as ptp_conversion
        ,case when sum(case when is_touch=1 then overdue_principal end)=0 then '/' else sum(case when is_touch=1 then overdue_principal-d11_principal end)/sum(case when is_touch=1 then overdue_principal end) end as disconnect_conversion
FROM    phl_anls.tmp_yuanpeng_phl_ana_09_eoc_lazada_sum_daily
WHERE to_char(due_date,'yyyymm') between '{start_duemon}' and '{end_duemon}'

GROUP BY 
GROUPING SETS (
            ()
              ,(case when user_type = '新客' then '新客'
            when user_type in ('新转化老客','存量老客') then '老客'
            else 'total' end)
              ,(case when user_type = '新客' then '新客'
            when user_type in ('新转化老客','存量老客') then '老客'
            else 'total' end,TO_CHAR(due_date,'yyyymm'))
            ,(case when user_type = '新客' then '新客'
            when user_type in ('新转化老客','存量老客') then '老客'
            else 'total' end, case when modelbinnew >='E' then 'E+' else modelbinnew end
                )
              ,(case when user_type = '新客' then '新客'
            when user_type in ('新转化老客','存量老客') then '老客'
            else 'total' end,flag_tt_principal)
            ,(mob)
            ,(case when user_type = '新客' then '新客'
            when user_type in ('新转化老客','存量老客') then '老客'
            else 'total' end,flag_tt_principal,to_char(due_date,'yyyymm'))
            ,(case when user_type = '新客' then '新客'
            when user_type in ('新转化老客','存量老客') then '老客'
            else 'total' end,case when modelbinnew >='E' then 'E+' else modelbinnew end,to_char(due_date,'yyyymm'))
        )
'''
df = run_sql(sql)

In [ ]:
df_total_lzd = df[(df['flag_tt_principal'].isna()) & (df['due_mon'].isna()) & (df['mob'].isna()) & (df['modelbin_a'].isna())].drop(columns=['due_mon','flag_tt_principal','mob']).sort_values(by='user_type')
df_duemon = df[df['due_mon'].notna()&df['flag_tt_principal'].isna()& (df['mob'].isna()) & (df['modelbin_a'].isna())].drop(columns=['flag_tt_principal','mob']).sort_values(by=['user_type', 'due_mon'],ascending=[True,False])
df_principal = df[df['flag_tt_principal'].notna()&df['due_mon'].isna()].drop(columns=['due_mon','mob']).sort_values(by=['user_type', 'flag_tt_principal'])
df_mob = df[(df['mob'].notna())].drop(columns=['flag_tt_principal','due_mon','user_type']).sort_values(by='mob')
df_modelbina = df[df['modelbin_a'].notna()&df['due_mon'].isna()].drop(columns=['flag_tt_principal','due_mon']).sort_values(by=['user_type','modelbin_a'])

df_principal_duemon = df[df['flag_tt_principal'].notna() & df['due_mon'].notna()].drop(columns='mob').groupby(by='user_type')
df_principal_duemon_user = df_principal_duemon.apply(lambda x: pd.pivot_table(x,index='flag_tt_principal',columns='due_mon',values='owing_user_cnt',aggfunc='sum'))
df_principal_duemon_user = df_principal_duemon_user.apply(lambda x:x.div(x.sum()))
df_principal_duemon_overdue = df_principal_duemon.apply(lambda x: pd.pivot_table(x,index='flag_tt_principal',columns='due_mon',values='overdue_rate',aggfunc='sum'))
df_principal_duemon_dpd5 = df_principal_duemon.apply(lambda x: pd.pivot_table(x,index='flag_tt_principal',columns='due_mon',values='dpd5',aggfunc='sum'))
df_principal_duemon_result = pd.concat([df_principal_duemon_user,df_principal_duemon_overdue,df_principal_duemon_dpd5],axis=1)

df_modelbina_duemon = df[(df['modelbin_a'].notna()) & (df['due_mon'].notna())].drop(columns='mob').sort_values('modelbin_a').groupby(by='user_type')
df_modelbina_duemon_user = df_modelbina_duemon.apply(lambda x: pd.pivot_table(x, index='modelbin_a', columns='due_mon', values='owing_user_cnt', aggfunc='sum'))
df_modelbina_duemon_user = df_modelbina_duemon_user.apply(lambda x: x.div(x.sum()))
df_modelbina_duemon_overdue = df_modelbina_duemon.apply(lambda x: pd.pivot_table(x,index='modelbin_a',columns='due_mon',values='overdue_rate',aggfunc='sum'))
df_modelbina_duemon_dpd5 = df_modelbina_duemon.apply(lambda x: pd.pivot_table(x,index='modelbin_a',columns='due_mon',values='dpd5',aggfunc='sum'))
df_modelbina_duemon_result = pd.concat([df_modelbina_duemon_user,df_modelbina_duemon_overdue,df_modelbina_duemon_dpd5],axis=1)

### 3.2+3 lzd过程指标

In [ ]:
# !!NEW!!
# user_type
sql = f'''
    SELECT  owner_bucket
            ,user_type
            
            ,sum(owner_cnt) / count(distinct dt) AS owner_cnt_avg
            ,SUM(case_cnt) / SUM(owner_cnt) AS owing_case_avg
            ,sum(comm_hour) / sum(owner_cnt) AS comm_hour_avg
            ,sum(call_case_cnt) / sum(case_cnt) as call_case_pct
            ,'' as call_duration_avg_owner
            ,SUM(call_times) / SUM(case_cnt) AS call_times_avg
            ,SUM(call_duration) / SUM(case_cnt) AS call_duration_avg
            ,SUM(call_contact_mobile_times) / sum(case_cnt) as call_contact_mobile_times_avg
            
            ,SUM(call_connect_times) / SUM(case_cnt) AS case_connect_times_avg
            ,SUM(connect_duration) / SUM(case_cnt) AS case_connect_duration_avg
            ,SUM(call_connect_case_cnt) / SUM(call_case_cnt) AS case_connect_rate
            ,SUM(call_connect_times) / SUM(call_times) AS call_connect_rate
            ,sum(call_connect_contact_mobile_times) / sum(call_contact_mobile_times) as contact_call_connect_rate
            ,SUM(connect_duration) / SUM(call_connect_case_cnt) as connect_duration_case_avg
    FROM    (
                SELECT  case when overdue_days_dt between 8 and 15 then 'L2'
                                when overdue_days_dt between 16 and 30 then 'LM1'
                                else owner_bucket
                                END AS owner_bucket
                        ,user_type
                        ,dt
                        ,datediff(CAST(max(last_comm_time) AS DATETIME), CAST(min(first_comm_time) AS DATETIME), 'hour')AS comm_hour
                        ,COUNT(DISTINCT CASE    WHEN art_call_times + batch_call_times > 0 THEN owner_id END) AS owner_cnt
                        ,COUNT(DISTINCT global_case_id) AS case_cnt
                        ,COUNT(DISTINCT global_case_id) / COUNT(DISTINCT owner_id) AS owing_case_avg
                        ,SUM(call_user_mobile_times + call_contact_mobile_times) AS call_times
                        ,SUM(call_user_mobile_ring_duration + call_user_mobile_billsec + call_contact_mobile_ring_duration + call_contact_mobile_billsec) / 60 AS call_duration
                        ,SUM(call_connect_user_mobile_times + call_connect_contact_mobile_times) AS call_connect_times
                        ,SUM(call_user_mobile_billsec + call_contact_mobile_billsec) / 60 AS connect_duration
                        ,SUM(call_contact_mobile_times) as call_contact_mobile_times
                        ,SUM(call_connect_contact_mobile_times) as call_connect_contact_mobile_times
                        ,COUNT(DISTINCT 
                              CASE    WHEN (art_call_connect_times + batch_call_connect_times) > 0 THEN global_case_id END
                        ) AS call_connect_case_cnt
                        ,COUNT(DISTINCT CASE    WHEN (art_call_times + batch_call_times) > 0 THEN global_case_id END) AS call_case_cnt
                        -- ,SUM(call_connect_user_mobile_times + call_connect_contact_mobile_times) / SUM(call_user_mobile_times + call_contact_mobile_times) AS call_connect_rate
                FROM    phl_data.dws_fact_coll_case_process_info_daily
                WHERE   dt >= '2024-10-20'
                AND     TO_CHAR(DATEADD(TO_DATE(dt),-overdue_days_dt,'day'),'yyyymm') BETWEEN '{start_duemon}' AND '{end_duemon}'
                AND     DAYOFWEEK(dt) <> 1 --排除周日
                AND     owner_bucket IN ('L0','L1','L2&3','L1&L2','L2&L3')
                GROUP BY case when overdue_days_dt between 8 and 15 then 'L2'
                                when overdue_days_dt between 16 and 30 then 'LM1'
                                else owner_bucket
                                END
                         ,user_type
                         ,dt
                         ,owner_id
                HAVING  SUM(art_call_times + batch_call_times) > 0
            ) a
    GROUP BY owner_bucket
             ,user_type
    ;
'''
df = run_sql(sql)
df_total_lzd_process = generate_bucket(df.sort_values('user_type'),['L0','L1','L2','LM1'])


df_total_lzd = pd.concat([df_total_lzd.reset_index(drop=True),df_total_lzd_process],axis=1)
df_total_lzd['tag'] = 'Lazada'

In [ ]:
# !!NEW!!
# user_type+due_mon
sql = f'''
    SELECT  owner_bucket
            ,user_type
            ,due_mon
            ,sum(owner_cnt) / count(distinct dt) AS owner_cnt_avg
            ,SUM(case_cnt) / SUM(owner_cnt) AS owing_case_avg
            ,sum(comm_hour) / sum(owner_cnt) AS comm_hour_avg
            ,sum(call_case_cnt) / sum(case_cnt) as call_case_pct
            ,SUM(call_duration) / sum(owner_cnt) as call_duration_avg_owner
            ,SUM(call_times) / SUM(case_cnt) AS call_times_avg
            ,SUM(call_duration) / SUM(case_cnt) AS call_duration_avg
            ,SUM(call_contact_mobile_times) / sum(case_cnt) as call_contact_mobile_times_avg


            
            ,SUM(call_connect_times) / SUM(case_cnt) AS case_connect_times_avg
            ,SUM(connect_duration) / SUM(case_cnt) AS case_connect_duration_avg
            ,SUM(call_connect_case_cnt) / SUM(call_case_cnt) AS case_connect_rate
            ,SUM(call_connect_times) / SUM(call_times) AS call_connect_rate
            ,sum(call_connect_contact_mobile_times) / sum(call_contact_mobile_times) as contact_call_connect_rate
            ,SUM(connect_duration) / SUM(call_connect_case_cnt) as connect_duration_case_avg
    FROM    (
                SELECT  case when overdue_days_dt between 8 and 15 then 'L2'
                                when overdue_days_dt between 16 and 30 then 'LM1'
                                else owner_bucket
                                END AS owner_bucket
                        ,user_type
                        ,dt
                        ,TO_CHAR(DATEADD(TO_DATE(dt),-overdue_days_dt,'day'),'yyyymm') as due_mon
                        ,datediff(CAST(max(last_comm_time) AS DATETIME), CAST(min(first_comm_time) AS DATETIME), 'hour')AS comm_hour
                        ,COUNT(DISTINCT CASE    WHEN art_call_times + batch_call_times > 0 THEN owner_id END) AS owner_cnt
                        ,COUNT(DISTINCT global_case_id) AS case_cnt
                        ,COUNT(DISTINCT global_case_id) / COUNT(DISTINCT owner_id) AS owing_case_avg
                        ,SUM(call_user_mobile_times + call_contact_mobile_times) AS call_times
                        ,SUM(call_user_mobile_ring_duration + call_user_mobile_billsec + call_contact_mobile_ring_duration + call_contact_mobile_billsec) / 60 AS call_duration
                        ,SUM(call_connect_user_mobile_times + call_connect_contact_mobile_times) AS call_connect_times
                        ,SUM(call_user_mobile_billsec + call_contact_mobile_billsec) / 60 AS connect_duration
                        ,SUM(call_contact_mobile_times) as call_contact_mobile_times
                        ,SUM(call_connect_contact_mobile_times) as call_connect_contact_mobile_times
                        ,COUNT(DISTINCT 
                              CASE    WHEN (art_call_connect_times + batch_call_connect_times) > 0 THEN global_case_id END
                        ) AS call_connect_case_cnt
                        ,COUNT(DISTINCT CASE    WHEN (art_call_times + batch_call_times) > 0 THEN global_case_id END) AS call_case_cnt
                        ,SUM(call_connect_user_mobile_times + call_connect_contact_mobile_times) / SUM(call_user_mobile_times + call_contact_mobile_times) AS call_connect_rate
                FROM    phl_data.dws_fact_coll_case_process_info_daily
                WHERE   dt >= '2024-10-20'
                AND     TO_CHAR(DATEADD(TO_DATE(dt),-overdue_days_dt,'day'),'yyyymm') BETWEEN '{start_duemon}' AND '{end_duemon}'
                AND     DAYOFWEEK(dt) <> 1 --排除周日
                AND     owner_bucket IN ('L0','L1','L2&3','L1&L2','L2&L3')
                GROUP BY case when overdue_days_dt between 8 and 15 then 'L2'
                                when overdue_days_dt between 16 and 30 then 'LM1'
                                else owner_bucket
                                END
                         ,user_type
                         ,dt
                         ,owner_id
                         ,TO_CHAR(DATEADD(TO_DATE(dt),-overdue_days_dt,'day'),'yyyymm')
                HAVING  SUM(art_call_times + batch_call_times) > 0
            ) a
    GROUP BY owner_bucket
             ,user_type
             ,due_mon
    ;
'''
df = run_sql(sql)
df_duemon_process = generate_bucket(df.sort_values(by=['user_type', 'due_mon'],ascending=[True,False]), ['L0','L1','L2','LM1'])

In [ ]:
# !!NEW!!
# user_type+flag_principal
sql = f'''
    SELECT  owner_bucket
            ,user_type
            ,flag_owing_principal_alloc
            ,sum(owner_cnt) / count(distinct dt) AS owner_cnt_avg
            ,SUM(case_cnt) / SUM(owner_cnt) AS owing_case_avg
            ,sum(comm_hour) / sum(owner_cnt) AS comm_hour_avg
            ,sum(call_case_cnt) / sum(case_cnt) as call_case_pct
            ,'' as call_duration_avg_owner
            ,SUM(call_times) / SUM(case_cnt) AS call_times_avg
            ,SUM(call_duration) / SUM(case_cnt) AS call_duration_avg
            ,SUM(call_contact_mobile_times) / sum(case_cnt) as call_contact_mobile_times_avg
            
            ,SUM(call_connect_times) / SUM(case_cnt) AS case_connect_times_avg
            ,SUM(connect_duration) / SUM(case_cnt) AS case_connect_duration_avg
            ,SUM(call_connect_case_cnt) / SUM(call_case_cnt) AS case_connect_rate
            ,SUM(call_connect_times) / SUM(call_times) AS call_connect_rate
            ,sum(call_connect_contact_mobile_times) / sum(call_contact_mobile_times) as contact_call_connect_rate
            ,SUM(connect_duration) / SUM(call_connect_case_cnt) as connect_duration_case_avg
    FROM    (
                SELECT  case when overdue_days_dt between 8 and 15 then 'L2'
                                when overdue_days_dt between 16 and 30 then 'LM1'
                                else owner_bucket
                                END AS owner_bucket
                        ,user_type
                        ,dt
                        ,datediff(CAST(max(last_comm_time) AS DATETIME), CAST(min(first_comm_time) AS DATETIME), 'hour')AS comm_hour
                        ,case when owing_principal_alloc < 500 then '1.0-500'
                            when owing_principal_alloc < 1000 then '2.500-1000'
                            when owing_principal_alloc < 1500 then '3.1000-1500'
                            when owing_principal_alloc < 2500 then '4.1500-2500'
                            when owing_principal_alloc < 3500 then '5.2500-3500'
                            else '6.3500+' end 
                            as flag_owing_principal_alloc
                        ,COUNT(DISTINCT CASE    WHEN art_call_times + batch_call_times > 0 THEN owner_id END) AS owner_cnt
                        ,COUNT(DISTINCT global_case_id) AS case_cnt
                        ,COUNT(DISTINCT global_case_id) / COUNT(DISTINCT owner_id) AS owing_case_avg
                        ,SUM(call_user_mobile_times + call_contact_mobile_times) AS call_times
                        ,SUM(call_user_mobile_ring_duration + call_user_mobile_billsec + call_contact_mobile_ring_duration + call_contact_mobile_billsec) / 60 AS call_duration
                        ,SUM(call_connect_user_mobile_times + call_connect_contact_mobile_times) AS call_connect_times
                        ,SUM(call_user_mobile_billsec + call_contact_mobile_billsec) / 60 AS connect_duration
                        ,SUM(call_contact_mobile_times) as call_contact_mobile_times
                        ,SUM(call_connect_contact_mobile_times) as call_connect_contact_mobile_times
                        ,COUNT(DISTINCT 
                              CASE    WHEN (art_call_connect_times + batch_call_connect_times) > 0 THEN global_case_id END
                        ) AS call_connect_case_cnt
                        ,COUNT(DISTINCT CASE    WHEN (art_call_times + batch_call_times) > 0 THEN global_case_id END) AS call_case_cnt
                        ,SUM(call_connect_user_mobile_times + call_connect_contact_mobile_times) / SUM(call_user_mobile_times + call_contact_mobile_times) AS call_connect_rate
                FROM    phl_data.dws_fact_coll_case_process_info_daily
                WHERE   dt >= '2024-10-20'
                AND     TO_CHAR(DATEADD(TO_DATE(dt),-overdue_days_dt,'day'),'yyyymm') BETWEEN '{start_duemon}' AND '{end_duemon}'
                AND     DAYOFWEEK(dt) <> 1   --排除周日
                AND     owner_bucket IN ('L0','L1','L2&3','L1&L2','L2&L3')
                GROUP BY case when overdue_days_dt between 8 and 15 then 'L2'
                                when overdue_days_dt between 16 and 30 then 'LM1'
                                else owner_bucket
                                END
                         ,user_type
                         ,dt
                         ,owner_id
                        ,case when owing_principal_alloc < 500 then '1.0-500'
                            when owing_principal_alloc < 1000 then '2.500-1000'
                            when owing_principal_alloc < 1500 then '3.1000-1500'
                            when owing_principal_alloc < 2500 then '4.1500-2500'
                            when owing_principal_alloc < 3500 then '5.2500-3500'
                            else '6.3500+' end 
                HAVING  SUM(art_call_times + batch_call_times) > 0
            ) a
    GROUP BY owner_bucket
             ,user_type
             ,flag_owing_principal_alloc
    ;
'''
df = run_sql(sql)
df_principal_process = generate_bucket(df.sort_values(by=['user_type', 'flag_owing_principal_alloc']), ['L0','L1','L2','LM1'])

### 3.4 lzd mob×过程指标

In [ ]:
# mob过程指标
sql=f'''
WITH tmp AS 
(
    SELECT  mob, modelbin_a
        ,a.*
    FROM   (
                    SELECT  DATEADD(TO_DATE(dt),-overdue_days_dt,'day') AS due_date
                            ,case when overdue_days_dt between 8 and 15 then 'L2'
                                when overdue_days_dt between 16 and 30 then 'LM1'
                                else owner_bucket
                                END AS owner_bucket
                            ,user_type
                            ,owner_id
                            ,borrower_id
                            ,global_case_id
                            ,dt
                            ,art_call_times
                            ,batch_call_times
                            ,call_user_mobile_times
                            ,call_contact_mobile_times
                            ,call_connect_user_mobile_times
                            ,call_connect_contact_mobile_times
                            ,call_user_mobile_ring_duration
                            ,call_contact_mobile_ring_duration
                            ,call_user_mobile_billsec
                            ,call_contact_mobile_billsec
                            ,first_comm_time
                            ,last_comm_time
                    FROM    phl_data.dws_fact_coll_case_process_info_daily
                    WHERE   dt >= '2024-10-20'
                    AND     TO_CHAR(DATEADD(TO_DATE(dt),-overdue_days_dt,'day'),'yyyymm') BETWEEN '{start_duemon}' AND '{end_duemon}'
                    AND     owner_bucket IN ('L0','L1','L2&3','L1&L2','L2&L3')
                ) a
    LEFT JOIN    (
            SELECT  user_id
                    ,due_date
                    ,mob_tt as mob
                    ,case when modelbinnew >='E' then 'E+' else modelbinnew end as modelbin_a
            FROM    phl_anls.tmp_zyt_lzd_vintage
            WHERE   TO_CHAR(due_date,'yyyymm')  BETWEEN '{start_duemon}' AND '{end_duemon}'
        ) v
    ON      v.user_id = a.borrower_id
    AND     v.due_date = a.due_date
)


SELECT  mob
        ,owner_bucket

            ,sum(owner_cnt) / count(distinct dt) AS owner_cnt_avg
            ,SUM(case_cnt) / SUM(owner_cnt) AS owing_case_avg
            ,sum(comm_hour) / sum(owner_cnt) AS comm_hour_avg
            ,sum(call_case_cnt) / sum(case_cnt) as call_case_pct
            ,'' as call_duration_avg_owner
            ,SUM(call_times) / SUM(case_cnt) AS call_times_avg
            ,SUM(call_duration) / SUM(case_cnt) AS call_duration_avg
            ,SUM(call_contact_mobile_times) / sum(case_cnt) as call_contact_mobile_times_avg


            
            ,SUM(call_connect_times) / SUM(case_cnt) AS case_connect_times_avg
            ,SUM(connect_duration) / SUM(case_cnt) AS case_connect_duration_avg
            ,SUM(call_connect_case_cnt) / SUM(call_case_cnt) AS case_connect_rate
            ,SUM(call_connect_times) / SUM(call_times) AS call_connect_rate
            ,sum(call_connect_contact_mobile_times) / sum(call_contact_mobile_times) as contact_call_connect_rate
            ,SUM(connect_duration) / SUM(call_connect_case_cnt) as connect_duration_case_avg
FROM    (
            SELECT  owner_bucket
                    ,mob

                    ,dt
                    ,datediff(CAST(max(last_comm_time) AS DATETIME), CAST(min(first_comm_time) AS DATETIME), 'hour')AS comm_hour
                    ,COUNT(DISTINCT CASE    WHEN art_call_times + batch_call_times > 0 THEN owner_id END) AS owner_cnt
                    ,COUNT(DISTINCT global_case_id) AS case_cnt
                    ,COUNT(DISTINCT global_case_id) / COUNT(DISTINCT owner_id) AS owing_case_avg
                    ,SUM(call_user_mobile_times + call_contact_mobile_times) AS call_times
                    ,SUM(call_user_mobile_ring_duration + call_user_mobile_billsec + call_contact_mobile_ring_duration + call_contact_mobile_billsec) / 60 AS call_duration
                    ,SUM(call_connect_user_mobile_times + call_connect_contact_mobile_times) AS call_connect_times
                    ,SUM(call_user_mobile_billsec + call_contact_mobile_billsec) / 60 AS connect_duration
                    ,SUM(call_contact_mobile_times) as call_contact_mobile_times
                    ,SUM(call_connect_contact_mobile_times) as call_connect_contact_mobile_times
                    ,COUNT(DISTINCT 
                          CASE    WHEN (call_connect_user_mobile_times + call_connect_contact_mobile_times) > 0 THEN global_case_id END
                    ) AS call_connect_case_cnt
                    ,COUNT(DISTINCT CASE    WHEN (art_call_times + batch_call_times) > 0 THEN global_case_id END) AS call_case_cnt
                    ,SUM(call_connect_user_mobile_times + call_connect_contact_mobile_times) / SUM(call_user_mobile_times + call_contact_mobile_times) AS call_connect_rate
            FROM    tmp
            GROUP BY owner_bucket
                        ,mob
                     ,dt
                     ,owner_id
            HAVING  SUM(art_call_times + batch_call_times) > 0 --拨打次数大于0 判断是否上班
        ) 
GROUP BY owner_bucket, mob
;
    '''
df = run_sql(sql)
df_mob_process = df[df['mob'].notna()].sort_values('mob')
df_mob_process = generate_bucket(df_mob_process, ['L0','L1','L2','LM1'])

In [ ]:
df_mob_process.to_excel('x.xlsx')

In [ ]:
# modelbin a过程指标
sql=f'''
WITH tmp AS 
(
    SELECT  mob, modelbin_a
        ,a.*
    FROM   (
                    SELECT  DATEADD(TO_DATE(dt),-overdue_days_dt,'day') AS due_date
                            ,case when overdue_days_dt between 8 and 15 then 'L2'
                                when overdue_days_dt between 16 and 30 then 'LM1'
                                else owner_bucket
                                END AS owner_bucket
                            ,user_type
                            ,owner_id
                            ,borrower_id
                            ,global_case_id
                            ,dt
                            ,art_call_times
                            ,batch_call_times
                            ,call_user_mobile_times
                            ,call_contact_mobile_times
                            ,call_connect_user_mobile_times
                            ,call_connect_contact_mobile_times
                            ,call_user_mobile_ring_duration
                            ,call_contact_mobile_ring_duration
                            ,call_user_mobile_billsec
                            ,call_contact_mobile_billsec
                            ,first_comm_time
                            ,last_comm_time
                    FROM    phl_data.dws_fact_coll_case_process_info_daily
                    WHERE   dt >= '2024-10-20'
                    AND     TO_CHAR(DATEADD(TO_DATE(dt),-overdue_days_dt,'day'),'yyyymm')  BETWEEN '{start_duemon}' AND '{end_duemon}'
                    AND     owner_bucket IN ('L0','L1','L2&3','L1&L2','L2&L3')
                ) a
    LEFT JOIN    (
            SELECT  user_id
                    ,due_date
                    ,mob_tt as mob
                    ,case when modelbinnew >='E' then 'E+' else modelbinnew end as modelbin_a
            FROM    phl_anls.tmp_zyt_lzd_vintage
            WHERE   TO_CHAR(due_date,'yyyymm')  BETWEEN '{start_duemon}' AND '{end_duemon}'
        ) v
    ON      v.user_id = a.borrower_id
    AND     substr(v.due_date,1,7) = substr(a.due_date,1,7)
)


SELECT modelbin_a
        ,owner_bucket
        ,user_type
        ,sum(owner_cnt) / count(distinct dt) AS owner_cnt_avg
        ,SUM(case_cnt) / SUM(owner_cnt) AS owing_case_avg
        ,sum(comm_hour) / sum(owner_cnt) AS comm_hour_avg
        ,sum(call_case_cnt) / sum(case_cnt) as call_case_pct
        ,'' as call_duration_avg_owner
        ,SUM(call_times) / SUM(case_cnt) AS call_times_avg
        ,SUM(call_duration) / SUM(case_cnt) AS call_duration_avg
        ,SUM(call_contact_mobile_times) / sum(case_cnt) as call_contact_mobile_times_avg



        ,SUM(call_connect_times) / SUM(case_cnt) AS case_connect_times_avg
        ,SUM(connect_duration) / SUM(case_cnt) AS case_connect_duration_avg
        ,SUM(call_connect_case_cnt) / SUM(call_case_cnt) AS case_connect_rate
        ,SUM(call_connect_times) / SUM(call_times) AS call_connect_rate
        ,sum(call_connect_contact_mobile_times) / sum(call_contact_mobile_times) as contact_call_connect_rate
        ,SUM(connect_duration) / SUM(call_connect_case_cnt) as connect_duration_case_avg
FROM    (
            SELECT  owner_bucket
                    ,modelbin_a
                    ,user_type
                    ,dt
                    ,datediff(CAST(max(last_comm_time) AS DATETIME), CAST(min(first_comm_time) AS DATETIME), 'hour')AS comm_hour
                    ,COUNT(DISTINCT CASE    WHEN art_call_times + batch_call_times > 0 THEN owner_id END) AS owner_cnt
                    ,COUNT(DISTINCT global_case_id) AS case_cnt
                    ,COUNT(DISTINCT global_case_id) / COUNT(DISTINCT owner_id) AS owing_case_avg
                    ,SUM(call_user_mobile_times + call_contact_mobile_times) AS call_times
                    ,SUM(call_user_mobile_ring_duration + call_user_mobile_billsec + call_contact_mobile_ring_duration + call_contact_mobile_billsec) / 60 AS call_duration
                    ,SUM(call_connect_user_mobile_times + call_connect_contact_mobile_times) AS call_connect_times
                    ,SUM(call_user_mobile_billsec + call_contact_mobile_billsec) / 60 AS connect_duration
                    ,SUM(call_contact_mobile_times) as call_contact_mobile_times
                    ,SUM(call_connect_contact_mobile_times) as call_connect_contact_mobile_times
                    ,COUNT(DISTINCT 
                          CASE    WHEN (call_connect_user_mobile_times + call_connect_contact_mobile_times) > 0 THEN global_case_id END
                    ) AS call_connect_case_cnt
                    ,COUNT(DISTINCT CASE    WHEN (art_call_times + batch_call_times) > 0 THEN global_case_id END) AS call_case_cnt
                    ,SUM(call_connect_user_mobile_times + call_connect_contact_mobile_times) / SUM(call_user_mobile_times + call_contact_mobile_times) AS call_connect_rate
            FROM    tmp
            GROUP BY owner_bucket, modelbin_a, user_type
                     ,dt
                     ,owner_id
            HAVING  SUM(art_call_times + batch_call_times) > 0 --拨打次数大于0 判断是否上班
        ) 
GROUP BY owner_bucket, user_type, modelbin_a
;
    '''
df = run_sql(sql)
df_modelbina_process = df[df['modelbin_a'].notna()].sort_values(['user_type','modelbin_a'])
df_modelbina_process = generate_bucket(df_modelbina_process, ['L0','L1','L2','LM1'])

In [ ]:
with pd.ExcelWriter('output.xlsx',mode='a') as writer:
    df_duemon.to_excel(writer, sheet_name='Lazada', startrow=2, startcol=0, index=True)
    df_duemon_process.to_excel(writer, sheet_name='Lazada', startrow=2, startcol=30, index=True)
    
    df_principal.to_excel(writer, sheet_name='Lazada', startrow=6+df_duemon.shape[0], startcol=0, index=True)
    df_principal_process.to_excel(writer, sheet_name='Lazada', startrow=6+df_duemon.shape[0], startcol=30, index=True)
    df_principal_duemon_result.to_excel(writer, sheet_name='Lazada', startrow=10+df_duemon.shape[0]*2, startcol=0, index=True)
    
    df_mob.to_excel(writer, sheet_name='Lazada', startrow=14+df_duemon.shape[0]*3, startcol=1, index=True)
    df_mob_process.to_excel(writer, sheet_name='Lazada', startrow=14+df_duemon.shape[0]*3, startcol=30, index=True)
    
    df_modelbina.to_excel(writer, sheet_name='Lazada', startrow=18+df_duemon.shape[0]*4, startcol=0, index=True)
    df_modelbina_process.to_excel(writer, sheet_name='Lazada', startrow=18+df_duemon.shape[0]*4, startcol=30, index=True)
    df_modelbina_duemon_result.to_excel(writer, sheet_name='Lazada', startrow=22+df_duemon.shape[0]*5, startcol=0, index=True)
    

In [ ]:
# df_total = pd.concat([df_total_cl,df_total_tt,df_total_lzd])
with pd.ExcelWriter('output.xlsx',mode='a') as writer:
    df_total_cl.to_excel(writer, sheet_name='Total', startrow=2, startcol=0, index=True)
    df_total_tt.to_excel(writer, sheet_name='Total', startrow=7, startcol=0, index=True)
    df_total_lzd.to_excel(writer, sheet_name='Total', startrow=12, startcol=0, index=True)